#  DRM + Wise-IoU + DWA vs. Epoch-Matched Control


- **Control**: unchanged architecture and loss (continues the paper's exact recipe) --
  isolates "does training longer alone help?"
- **Proposed**: DRM (Detail Recovery Module at P3) + Wise-IoU dynamic focusing + Dynamic
  Weight Averaging (DWA) for the detection/dehaze loss balance -- isolates "does the
  technique help beyond that?"

Both are then evaluated on **VOC-FOG-test** and **RTTS** (FDD intentionally out of scope),
alongside the unmodified baseline checkpoint for reference, using the same subprocess-isolated
`eval_one.py` for every checkpoint so each evaluation always builds the exact architecture
(`USE_DRM`) that checkpoint was trained with -- see `eval_one.py`'s docstring for why this
matters (Python caches module imports; DRM changes `nets/model.py`'s architecture at import
time based on `config.USE_DRM`).

### Before running
1. In the right sidebar, click **+ Add Data** and attach:
   - `mdhabibourrahman/voc-fog`
   - `tuncnguyn/rtts-dataset`
   - `mdhabibourrahman/rdfnet-baseline`
2. **Settings > Accelerator > GPU T4 x2** -- required exactly as-is: `nets/backbone.py`
   hardcodes an `x.split((8, 8), dim=0)` that assumes a 2-GPU DDP split of a
   batch-16-hazy + batch-16-clean pair into 8+8 per GPU. Do not change batch size or GPU
   count without also fixing that hardcoded split.
3. Run all cells top to bottom. **Read the "Set the fine-tune epoch budget" cell before
   launching training** -- the default (`FINETUNE_EPOCHS = 60`) is a placeholder pending your
   own timing calibration; the cell explains how to adjust it after watching the first
   epoch's progress bar.
4. Each training run can take hours; if a Kaggle session times out mid-run, checkpoints are
   saved every `save_period` epochs under `logs_control/` / `logs_proposed/` -- re-attach this
   notebook's `/kaggle/working` output as an input dataset in a new session to resume manually
   (not automated here).

In [ ]:
%cd /kaggle/working
!rm -rf adverse_weather_ibject_detection
!git clone --depth 1 https://github.com/habibour/adverse_weather_ibject_detection.git
%cd /kaggle/working/adverse_weather_ibject_detection/RDFNet

# Verify exactly which commit Kaggle actually cloned -- compare this against
# the latest commit hash on github.com/habibour/adverse_weather_ibject_detection
# to confirm a manually re-uploaded notebook is running against fresh code.
!git log -1 --format='Cloned commit: %H%nCommit date:   %ci%nSubject:       %s'

## Apply this thesis's modifications (self-contained, no git push required)

Writes the 6 changed/new files (DRM, Wise-IoU, DWA, config toggles, mAP-based checkpoint
selection, `eval_one.py`) directly on top of the freshly cloned base repo. This does not
depend on `github.com/habibour/adverse_weather_ibject_detection` having been updated --
everything needed is embedded in this notebook.

In [ ]:
import base64, os

_FILES_B64 = {
    'nets/Common.py': (
        'aW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgpmcm9tIHRob3AgaW1wb3J0IHByb2ZpbGUgCgpjbGFzcyBTaUxVKG5uLk1vZHVsZS'
        'k6CiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgZm9yd2FyZCh4KToKICAgICAgICByZXR1cm4geCAqIHRvcmNoLnNpZ21vaWQoeCkKCmRlZiBh'
        'dXRvcGFkKGssIHA9Tm9uZSk6CiAgICBpZiBwIGlzIE5vbmU6CiAgICAgICAgcCA9IGsgLy8gMiBpZiBpc2luc3RhbmNlKGssIGludCkgZWxzZS'
        'BbeCAvLyAyIGZvciB4IGluIGtdIAogICAgcmV0dXJuIHAKICAgIApjbGFzcyBDb252KG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2Vs'
        'ZiwgYzEsIGMyLCBrPTEsIHM9MSwgcD1Ob25lLCBnPTEsIGFjdD1ubi5MZWFreVJlTFUoMC4xLCBpbnBsYWNlPVRydWUpKTogICMgY2hfaW4sIG'
        'NoX291dCwga2VybmVsLCBzdHJpZGUsIHBhZGRpbmcsIGdyb3VwcwogICAgICAgIHN1cGVyKENvbnYsIHNlbGYpLl9faW5pdF9fKCkKICAgICAg'
        'ICBzZWxmLmNvbnYgICA9IG5uLkNvbnYyZChjMSwgYzIsIGssIHMsIGF1dG9wYWQoaywgcCksIGdyb3Vwcz1nLCBiaWFzPUZhbHNlKQogICAgIC'
        'AgIHNlbGYuYm4gICAgID0gbm4uQmF0Y2hOb3JtMmQoYzIsIGVwcz0wLjAwMSwgbW9tZW50dW09MC4wMykKICAgICAgICBzZWxmLmFjdCAgICA9'
        'IG5uLkxlYWt5UmVMVSgwLjEsIGlucGxhY2U9VHJ1ZSkgaWYgYWN0IGlzIFRydWUgZWxzZSAoYWN0IGlmIGlzaW5zdGFuY2UoYWN0LCBubi5Nb2'
        'R1bGUpIGVsc2Ugbm4uSWRlbnRpdHkoKSkKCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgcmV0dXJuIHNlbGYuYWN0KHNlbGYu'
        'Ym4oc2VsZi5jb252KHgpKSkKCiAgICBkZWYgZnVzZWZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgcmV0dXJuIHNlbGYuYWN0KHNlbGYuY29udi'
        'h4KSkKICAgIApjbGFzcyBCYXNpY0NvbnYobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGluX3Bs'
        'YW5lcywKICAgICAgICBvdXRfcGxhbmVzLAogICAgICAgIGtlcm5lbF9zaXplLAogICAgICAgIHN0cmlkZT0xLAogICAgICAgIHBhZGRpbmc9MC'
        'wKICAgICAgICBkaWxhdGlvbj0xLAogICAgICAgIGdyb3Vwcz0xLAogICAgICAgIHJlbHU9VHJ1ZSwKICAgICAgICBibj1UcnVlLAogICAgICAg'
        'IGJpYXM9RmFsc2UsCiAgICApOgogICAgICAgIHN1cGVyKEJhc2ljQ29udiwgc2VsZikuX19pbml0X18oKQogICAgICAgIHNlbGYub3V0X2NoYW'
        '5uZWxzID0gb3V0X3BsYW5lcwogICAgICAgIHNlbGYuY29udiA9IG5uLkNvbnYyZCgKICAgICAgICAgICAgaW5fcGxhbmVzLAogICAgICAgICAg'
        'ICBvdXRfcGxhbmVzLAogICAgICAgICAgICBrZXJuZWxfc2l6ZT1rZXJuZWxfc2l6ZSwKICAgICAgICAgICAgc3RyaWRlPXN0cmlkZSwKICAgIC'
        'AgICAgICAgcGFkZGluZz1wYWRkaW5nLAogICAgICAgICAgICBkaWxhdGlvbj1kaWxhdGlvbiwKICAgICAgICAgICAgZ3JvdXBzPWdyb3VwcywK'
        'ICAgICAgICAgICAgYmlhcz1iaWFzLAogICAgICAgICkKICAgICAgICBzZWxmLmJuID0gKAogICAgICAgICAgICBubi5CYXRjaE5vcm0yZChvdX'
        'RfcGxhbmVzLCBlcHM9MWUtNSwgbW9tZW50dW09MC4wMSwgYWZmaW5lPVRydWUpCiAgICAgICAgICAgIGlmIGJuCiAgICAgICAgICAgIGVsc2Ug'
        'Tm9uZQogICAgICAgICkKICAgICAgICBzZWxmLnJlbHUgPSBubi5SZUxVKCkgaWYgcmVsdSBlbHNlIE5vbmUKCiAgICBkZWYgZm9yd2FyZChzZW'
        'xmLCB4KToKICAgICAgICB4ID0gc2VsZi5jb252KHgpCiAgICAgICAgaWYgc2VsZi5ibiBpcyBub3QgTm9uZToKICAgICAgICAgICAgeCA9IHNl'
        'bGYuYm4oeCkKICAgICAgICBpZiBzZWxmLnJlbHUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHggPSBzZWxmLnJlbHUoeCkKICAgICAgICByZX'
        'R1cm4geAoKCmNsYXNzIENoYW5uZWxQb29sKG5uLk1vZHVsZSk6CiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICByZXR1cm4gdG9y'
        'Y2guY2F0KAogICAgICAgICAgICAodG9yY2gubWF4KHgsIDEpWzBdLnVuc3F1ZWV6ZSgxKSwgdG9yY2gubWVhbih4LCAxKS51bnNxdWVlemUoMS'
        'kpLCBkaW09MQogICAgICAgICkKCgpjbGFzcyBTcGF0aWFsR2F0ZShubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYpOgogICAgICAg'
        'IHN1cGVyKFNwYXRpYWxHYXRlLCBzZWxmKS5fX2luaXRfXygpCiAgICAgICAga2VybmVsX3NpemUgPSA3CiAgICAgICAgc2VsZi5jb21wcmVzcy'
        'A9IENoYW5uZWxQb29sKCkKICAgICAgICBzZWxmLnNwYXRpYWwgPSBCYXNpY0NvbnYoCiAgICAgICAgICAgIDIsIDEsIGtlcm5lbF9zaXplLCBz'
        'dHJpZGU9MSwgcGFkZGluZz0oa2VybmVsX3NpemUgLSAxKSAvLyAyLCByZWx1PUZhbHNlCiAgICAgICAgKQoKICAgIGRlZiBmb3J3YXJkKHNlbG'
        'YsIHgpOgogICAgICAgIHhfY29tcHJlc3MgPSBzZWxmLmNvbXByZXNzKHgpCiAgICAgICAgeF9vdXQgPSBzZWxmLnNwYXRpYWwoeF9jb21wcmVz'
        'cykKICAgICAgICBzY2FsZSA9IHRvcmNoLnNpZ21vaWRfKHhfb3V0KQogICAgICAgIHJldHVybiB4ICogc2NhbGUKCmRlZiBhdXRvcGFkKGssIH'
        'A9Tm9uZSk6CiAgICBpZiBwIGlzIE5vbmU6CiAgICAgICAgcCA9IGsgLy8gMiBpZiBpc2luc3RhbmNlKGssIGludCkgZWxzZSBbeCAvLyAyIGZv'
        'ciB4IGluIGtdIAogICAgcmV0dXJuIHAKICAgIApjbGFzcyBDb252KG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgYzEsIGMyLC'
        'BrPTEsIHM9MSwgcD1Ob25lLCBnPTEsIGFjdD1ubi5MZWFreVJlTFUoMC4xLCBpbnBsYWNlPVRydWUpKTogICMgY2hfaW4sIGNoX291dCwga2Vy'
        'bmVsLCBzdHJpZGUsIHBhZGRpbmcsIGdyb3VwcwogICAgICAgIHN1cGVyKENvbnYsIHNlbGYpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbn'
        'YgICA9IG5uLkNvbnYyZChjMSwgYzIsIGssIHMsIGF1dG9wYWQoaywgcCksIGdyb3Vwcz1nLCBiaWFzPUZhbHNlKQogICAgICAgIHNlbGYuYm4g'
        'ICAgID0gbm4uQmF0Y2hOb3JtMmQoYzIsIGVwcz0wLjAwMSwgbW9tZW50dW09MC4wMykKICAgICAgICBzZWxmLmFjdCAgICA9IG5uLkxlYWt5Um'
        'VMVSgwLjEsIGlucGxhY2U9VHJ1ZSkgaWYgYWN0IGlzIFRydWUgZWxzZSAoYWN0IGlmIGlzaW5zdGFuY2UoYWN0LCBubi5Nb2R1bGUpIGVsc2Ug'
        'bm4uSWRlbnRpdHkoKSkKCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgcmV0dXJuIHNlbGYuYWN0KHNlbGYuYm4oc2VsZi5jb2'
        '52KHgpKSkKI2xpZ2h0aW5nIGRlaGF6ZSBuZXR3b3JrCmNsYXNzIExNRE5ldChubi5Nb2R1bGUpOgoKICAgIGRlZiBfX2luaXRfXyhzZWxmKToK'
        'ICAgICAgICBzdXBlcihMTUROZXQsIHNlbGYpLl9faW5pdF9fKCkKICAgICAgICAjIG1haW5OZXQgQXJjaGl0ZWN0dXJlCiAgICAgICAgc2VsZi'
        '5BQU0gPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5Db252MmQoNjQsIDMsIDEsIDEpLAogICAgICAgICAgICBubi5MZWFreVJlTFUo'
        'aW5wbGFjZT1UcnVlKSwKICAgICAgICAgICAgbm4uRHJvcG91dCgwLjUpCiAgICAgICAgKQogICAgICAgIHNlbGYuQUFNXzEgPSBubi5TZXF1ZW'
        '50aWFsKAogICAgICAgICAgICBubi5VcHNhbXBsZShzY2FsZV9mYWN0b3I9MiwgbW9kZT0nYmlsaW5lYXInLCBhbGlnbl9jb3JuZXJzPVRydWUp'
        'LAogICAgICAgICAgICBubi5Db252MmQoMTI4LCAzMiwgMSwgMSksCiAgICAgICAgICAgIG5uLkxlYWt5UmVMVShpbnBsYWNlPVRydWUpLAogIC'
        'AgICAgICAgICBubi5Ecm9wb3V0KDAuNSkKICAgICAgICApCiAgICAgICAgc2VsZi5BQU1fMiA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAg'
        'IG5uLlVwc2FtcGxlKHNjYWxlX2ZhY3Rvcj00LCBtb2RlPSdiaWxpbmVhcicsIGFsaWduX2Nvcm5lcnM9VHJ1ZSksCiAgICAgICAgICAgIG5uLk'
        'NvbnYyZCgyNTYsIDMyLCAxLCAxKSwKICAgICAgICAgICAgbm4uTGVha3lSZUxVKGlucGxhY2U9VHJ1ZSksCiAgICAgICAgICAgIG5uLkRyb3Bv'
        'dXQoMC41KQogICAgICAgICkKICAgICAgICBzZWxmLlRBID0gVHJpcGxldEF0dGVudGlvbig2NCkKCiAgICAgICAgc2VsZi5jb252ID0gQ29udi'
        'g2NCwgMywgMywgMSkKCiAgICAgICAgc2VsZi51cDQgPSBubi5VcHNhbXBsZShzY2FsZV9mYWN0b3I9NCwgbW9kZT0nYmlsaW5lYXInLCBhbGln'
        'bl9jb3JuZXJzPVRydWUpCiAgICAgICAgc2VsZi5yZWx1ID0gbm4uTGVha3lSZUxVKDAuMSwgaW5wbGFjZT1UcnVlKQoKCiAgICBkZWYgZm9yd2'
        'FyZChzZWxmLCBmMSwgZjIsIGYzKToKICAgICAgICB0ID0gc2VsZi5BQU0oZjEpIAogICAgICAgIGYyID0gc2VsZi5BQU1fMShmMikKICAgICAg'
        'ICBmMyA9IHNlbGYuQUFNXzIoZjMpCiAgICAgICAgeDEgPSBmMQogICAgICAgIHgyID0gdG9yY2guY2F0KFtmMiwgZjNdLCBkaW09MSkKICAgIC'
        'AgICAKICAgICAgICB4ID0geDEgKyB4MgogICAgICAgIHggPSBzZWxmLlRBKHgpCiAgICAgICAgeCA9IHNlbGYuY29udih4KQoKICAgICAgICBk'
        'ZWhhemUgPSAoKHggKiB0KSAtIHggKyAxKQoKICAgICAgICBvdXQgPSBzZWxmLnVwNChkZWhhemUpCiAgICAgICAgb3V0ID0gc2VsZi5yZWx1KG'
        '91dCkgICAKCiAgICAgICAgcmV0dXJuIG91dAogICAgICAgIApjbGFzcyBUcmlwbGV0QXR0ZW50aW9uKG5uLk1vZHVsZSk6CiAgICBkZWYgX19p'
        'bml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBpbl9jaGFubmVscywKICAgICAgICByZWR1Y3Rpb25fcmF0aW89MTYsCiAgICAgICAgcG9vbF'
        '90eXBlcz1bImF2ZyIsICJtYXgiXSwKICAgICAgICBub19zcGF0aWFsPUZhbHNlLAogICAgKToKICAgICAgICBzdXBlcihUcmlwbGV0QXR0ZW50'
        'aW9uLCBzZWxmKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5DaGFubmVsR2F0ZUggPSBTcGF0aWFsR2F0ZSgpCiAgICAgICAgc2VsZi5DaGFubm'
        'VsR2F0ZVcgPSBTcGF0aWFsR2F0ZSgpCiAgICAgICAgc2VsZi5ub19zcGF0aWFsID0gbm9fc3BhdGlhbAogICAgICAgIGlmIG5vdCBub19zcGF0'
        'aWFsOgogICAgICAgICAgICBzZWxmLlNwYXRpYWxHYXRlID0gU3BhdGlhbEdhdGUoKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgIC'
        'AgIHhfcGVybTEgPSB4LnBlcm11dGUoMCwgMiwgMSwgMykuY29udGlndW91cygpCiAgICAgICAgeF9vdXQxID0gc2VsZi5DaGFubmVsR2F0ZUgo'
        'eF9wZXJtMSkKICAgICAgICB4X291dDExID0geF9vdXQxLnBlcm11dGUoMCwgMiwgMSwgMykuY29udGlndW91cygpCiAgICAgICAgeF9wZXJtMi'
        'A9IHgucGVybXV0ZSgwLCAzLCAyLCAxKS5jb250aWd1b3VzKCkKICAgICAgICB4X291dDIgPSBzZWxmLkNoYW5uZWxHYXRlVyh4X3Blcm0yKQog'
        'ICAgICAgIHhfb3V0MjEgPSB4X291dDIucGVybXV0ZSgwLCAzLCAyLCAxKS5jb250aWd1b3VzKCkKICAgICAgICBpZiBub3Qgc2VsZi5ub19zcG'
        'F0aWFsOgogICAgICAgICAgICB4X291dCA9IHNlbGYuU3BhdGlhbEdhdGUoeCkKICAgICAgICAgICAgeF9vdXQgPSAoMSAvIDMpICogKHhfb3V0'
        'ICsgeF9vdXQxMSArIHhfb3V0MjEpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgeF9vdXQgPSAoMSAvIDIpICogKHhfb3V0MTEgKyB4X291dD'
        'IxKQogICAgICAgIHJldHVybiB4X291dAoKY2xhc3MgU2lMVShubi5Nb2R1bGUpOgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIGZvcndhcmQo'
        'eCk6CiAgICAgICAgcmV0dXJuIHggKiB0b3JjaC5zaWdtb2lkKHgpCgpkZWYgYXV0b3BhZChrLCBwPU5vbmUpOgogICAgaWYgcCBpcyBOb25lOg'
        'ogICAgICAgIHAgPSBrIC8vIDIgaWYgaXNpbnN0YW5jZShrLCBpbnQpIGVsc2UgW3ggLy8gMiBmb3IgeCBpbiBrXSAKICAgIHJldHVybiBwCgpj'
        'bGFzcyBDb252KG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgYzEsIGMyLCBrPTEsIHM9MSwgcD1Ob25lLCBnPTEsIGFjdD1ubi'
        '5MZWFreVJlTFUoMC4xLCBpbnBsYWNlPVRydWUpKTogIAogICAgICAgIHN1cGVyKENvbnYsIHNlbGYpLl9faW5pdF9fKCkKICAgICAgICBzZWxm'
        'LmNvbnYgICA9IG5uLkNvbnYyZChjMSwgYzIsIGssIHMsIGF1dG9wYWQoaywgcCksIGdyb3Vwcz1nLCBiaWFzPUZhbHNlKQogICAgICAgIHNlbG'
        'YuYm4gICAgID0gbm4uQmF0Y2hOb3JtMmQoYzIsIGVwcz0wLjAwMSwgbW9tZW50dW09MC4wMykKICAgICAgICBzZWxmLmFjdCAgICA9IG5uLkxl'
        'YWt5UmVMVSgwLjEsIGlucGxhY2U9VHJ1ZSkgaWYgYWN0IGlzIFRydWUgZWxzZSAoYWN0IGlmIGlzaW5zdGFuY2UoYWN0LCBubi5Nb2R1bGUpIG'
        'Vsc2Ugbm4uSWRlbnRpdHkoKSkKCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgcmV0dXJuIHNlbGYuYWN0KHNlbGYuYm4oc2Vs'
        'Zi5jb252KHgpKSkKCiAgICBkZWYgZnVzZWZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgcmV0dXJuIHNlbGYuYWN0KHNlbGYuY29udih4KSkKIC'
        'AgIApjbGFzcyBHSUUodG9yY2gubm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBlcHNpbG9uPTFlLTgpOgogICAgICAgIHN1cGVy'
        'KEdJRSwgc2VsZikuX19pbml0X18oKQogICAgICAgIHNlbGYuZXBzaWxvbiA9IGVwc2lsb24KCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKIC'
        'AgICAgICAjIFN0ZXAgMTogUGl4ZWwgTWVhbiBTcXVhcmVkCiAgICAgICAgeF9tZWFuID0gdG9yY2gubWVhbih4LCBkaW09KDIsIDMpLCBrZWVw'
        'ZGltPVRydWUpIAogICAgICAgIGVwc2lsb24gPSAoeCAtIHhfbWVhbikgKiogMiAgCiAgICAgICAgIyBTdGVwIDI6IEdsb2JhbCBFeHRyYWN0aW'
        '9uCiAgICAgICAgZXBzaWxvbl9tZWFuID0gdG9yY2gubWVhbihlcHNpbG9uLCBkaW09KDIsIDMpLCBrZWVwZGltPUZhbHNlKSAgCiAgICAgICAg'
        'ZXBzaWxvbl9tZWFuICs9IHNlbGYuZXBzaWxvbgogICAgICAgICMgU3RlcCAzOiBHYW1tYSBDYWxjdWxhdGlvbiAoR2xvYmFsIEV4dHJhY3Rpb2'
        '4pCiAgICAgICAgZ2FtbWFfdF9jID0gZXBzaWxvbiAvIGVwc2lsb25fbWVhbi51bnNxdWVlemUoLTEpLnVuc3F1ZWV6ZSgtMSkgIAogICAgICAg'
        'IHNpZ21vaWRfZ2FtbWEgPSB0b3JjaC5zaWdtb2lkKGdhbW1hX3RfYykgCiAgICAgICAgb3V0cHV0ID0geCAqIHNpZ21vaWRfZ2FtbWEgCiAgIC'
        'AgICAgcmV0dXJuIG91dHB1dAogICAgCiMgTXVsdGktYnJhbmNoIFBvb2xpbmcgSW5mb3JtYXRpb24gRnVzaW9uIE1vZHVsZQpjbGFzcyBNUElG'
        'KG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgYzEsIGMyLCBjMywgcz0yLCBuPTQsIGU9MSwgaWRzPVswXSk6CiAgICAgICAgc3'
        'VwZXIoTVBJRiwgc2VsZikuX19pbml0X18oKQogICAgICAgIGNfID0gaW50KGMyICogZSkKICAgICAgICAKICAgICAgICBzZWxmLmlkcyA9IGlk'
        'cwogICAgICAgIGlmIHMgPT0gMToKICAgICAgICAgICAgc2VsZi5tMSA9IG5uLk1heFBvb2wyZChrZXJuZWxfc2l6ZT0zLCBzdHJpZGU9cywgcG'
        'FkZGluZz0xKQogICAgICAgICAgICBzZWxmLm0yID0gbm4uQXZnUG9vbDJkKGtlcm5lbF9zaXplPTMsIHN0cmlkZT1zLCBwYWRkaW5nPTEpCiAg'
        'ICAgICAgZWxzZToKICAgICAgICAgICAgc2VsZi5tMSA9IG5uLk1heFBvb2wyZChrZXJuZWxfc2l6ZT0yLCBzdHJpZGU9cykKICAgICAgICAgIC'
        'Agc2VsZi5tMiA9IG5uLkF2Z1Bvb2wyZChrZXJuZWxfc2l6ZT0yLCBzdHJpZGU9cykKICAgICAgICAKICAgICAgICBzZWxmLmN2MSA9IENvbnYo'
        'YzEsIGNfLCAxLCAxKQogICAgICAgIHNlbGYuY3YyID0gQ29udihjMSwgY18sIDEsIDEpCiAgICAgICAgc2VsZi5jdjMgPSBubi5Nb2R1bGVMaX'
        'N0KAogICAgICAgICAgICBbQ29udihjXyBpZiBpID09MCBlbHNlIGMyLCBjMiwgMywgMSkgZm9yIGkgaW4gcmFuZ2UobildCiAgICAgICAgKQog'
        'ICAgICAgIHNlbGYuY3Y0ID0gQ29udihjXyAqIDIgKyBjMiAqIChsZW4oaWRzKSAtIDIpLCBjMywgMSwgMSkKCiAgICAgICAgc2VsZi5HSUUgPS'
        'BHSUUoYzEpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgeDEgPSBzZWxmLm0xKHgpCiAgICAgICAgeDIgPSBzZWxmLm0yKHgp'
        'CiAgICAgICAgeCA9IHgxICsgeDIKICAgICAgICB4XzEgPSBzZWxmLmN2MSh4KQogICAgICAgIHhfMSA9IHNlbGYuR0lFKHhfMSkKICAgICAgIC'
        'B4XzIgPSBzZWxmLmN2Mih4KQogICAgICAgIAogICAgICAgIHhfYWxsID0gW3hfMSwgeF8yXQoKICAgICAgICBmb3IgaSBpbiByYW5nZShsZW4o'
        'c2VsZi5jdjMpKToKICAgICAgICAgICAgeF8yID0gc2VsZi5jdjNbaV0oeF8yKQogICAgICAgICAgICB4X2FsbC5hcHBlbmQoeF8yKQogICAgIC'
        'AgIAogICAgICAgIG91dCA9IHNlbGYuY3Y0KHRvcmNoLmNhdChbeF9hbGxbaWRdIGZvciBpZCBpbiBzZWxmLmlkc10sIDEpKQoKICAgICAgICBy'
        'ZXR1cm4gb3V0CiAgICAKY2xhc3MgRGlsYXRlZENvbnZOZXQobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbl9jaGFubmVscy'
        'wgb3V0X2NoYW5uZWxzLCBkaWxhdGlvbiwgcGFkZGluZywga2VybmVsX3NpemUpOgogICAgICAgIHN1cGVyKERpbGF0ZWRDb252TmV0LCBzZWxm'
        'KS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5kaWxhdGVkX2NvbnYgPSBubi5Db252MmQoaW5fY2hhbm5lbHMsIG91dF9jaGFubmVscywga2Vybm'
        'VsX3NpemU9a2VybmVsX3NpemUsIHN0cmlkZT0xLCBwYWRkaW5nPXBhZGRpbmcsIGRpbGF0aW9uPWRpbGF0aW9uKQogICAgICAgIHNlbGYucmVs'
        'dSA9IG5uLlJlTFUoaW5wbGFjZT1GYWxzZSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKCiAgICAgICAgeCA9IHNlbGYuZGlsYXRlZF9jb2'
        '52KHgpCiAgICAgICAgeCA9IHNlbGYucmVsdSh4KQoKICAgICAgICByZXR1cm4geAogICAgCmNsYXNzIFNQUEVMQU4obm4uTW9kdWxlKToKICAg'
        'IGRlZiBfX2luaXRfXyhzZWxmLCBjMSwgYzIsIGMzPTE2KTogIAogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuYyA9IG'
        'MzCiAgICAgICAgc2VsZi5jdjEgPSBDb252KGMxLCBjMywgMSwgMSkKICAgICAgICBzZWxmLmN2MiA9IERpbGF0ZWRDb252TmV0KGMzLCBjMywg'
        'a2VybmVsX3NpemU9MywgcGFkZGluZz0xLCBkaWxhdGlvbj0xKQogICAgICAgIHNlbGYuY3YzID0gRGlsYXRlZENvbnZOZXQoYzMsIGMzLCBrZX'
        'JuZWxfc2l6ZT0zLCBwYWRkaW5nPTIsIGRpbGF0aW9uPTIpCiAgICAgICAgc2VsZi5jdjQgPSBEaWxhdGVkQ29udk5ldChjMywgYzMsIGtlcm5l'
        'bF9zaXplPTMsIHBhZGRpbmc9MywgZGlsYXRpb249MykKICAgICAgICBzZWxmLmN2NSA9IENvbnYoNCpjMywgYzIsIDEsIDEpCgogICAgZGVmIG'
        'ZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgeSA9IFtzZWxmLmN2MSh4KV0KICAgICAgICB5LmV4dGVuZChtKHlbLTFdKSBmb3IgbSBpbiBbc2Vs'
        'Zi5jdjIsIHNlbGYuY3YzLCBzZWxmLmN2NF0pCiAgICAgICAgcmV0dXJuIHNlbGYuY3Y1KHRvcmNoLmNhdCh5LCAxKSkKICAgICAgICAKCgpjbG'
        'FzcyBFQ0Eobm4uTW9kdWxlKToKICAgICIiIkVmZmljaWVudCBDaGFubmVsIEF0dGVudGlvbiAoV2FuZyBldCBhbC4sIENWUFIgMjAyMCkgLS0g'
        'bmVhci16ZXJvLXBhcmFtZXRlcgogICAgY2hhbm5lbCBhdHRlbnRpb24gdmlhIGEgMUQgY29udm9sdXRpb24gb3ZlciBnbG9iYWxseSBwb29sZW'
        'QgY2hhbm5lbCBkZXNjcmlwdG9ycy4KICAgIFVzZWQgaW5zaWRlIERSTSB0byBjaGVhcGx5IHJlY2FsaWJyYXRlIHdoaWNoIGNoYW5uZWxzIG1h'
        'dHRlciBhZnRlciB0aGUKICAgIGRldGFpbC1yZWNvdmVyeSBicmFuY2hlcywgd2l0aG91dCB0aGUgY29zdCBvZiBhIGZ1bGwgc3F1ZWV6ZS1leG'
        'NpdGUgTUxQLiIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNoYW5uZWxzLCBrX3NpemU9Myk6CiAgICAgICAgc3VwZXIoRUNBLCBzZWxmKS5f'
        'X2luaXRfXygpCiAgICAgICAgc2VsZi5hdmdfcG9vbCA9IG5uLkFkYXB0aXZlQXZnUG9vbDJkKDEpCiAgICAgICAgc2VsZi5jb252ICAgICA9IG'
        '5uLkNvbnYxZCgxLCAxLCBrZXJuZWxfc2l6ZT1rX3NpemUsIHBhZGRpbmc9KGtfc2l6ZSAtIDEpIC8vIDIsIGJpYXM9RmFsc2UpCiAgICAgICAg'
        'c2VsZi5zaWdtb2lkICA9IG5uLlNpZ21vaWQoKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIHkgPSBzZWxmLmF2Z19wb29sKH'
        'gpICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIChCLCBDLCAxLCAxKQogICAgICAgIHkgPSB5LnNxdWVlemUoLTEpLnRyYW5zcG9zZSgt'
        'MSwgLTIpICAgICAgICAgICAgICAgICMgKEIsIDEsIEMpCiAgICAgICAgeSA9IHNlbGYuY29udih5KSAgICAgICAgICAgICAgICAgICAgICAgIC'
        'AgICAgICAgICAgIyAoQiwgMSwgQykKICAgICAgICB5ID0geS50cmFuc3Bvc2UoLTEsIC0yKS51bnNxdWVlemUoLTEpICAgICAgICAgICAgICAg'
        'IyAoQiwgQywgMSwgMSkKICAgICAgICB5ID0gc2VsZi5zaWdtb2lkKHkpCiAgICAgICAgcmV0dXJuIHggKiB5CgoKY2xhc3MgRGVwdGh3aXNlRG'
        'lsYXRlZENvbnYobm4uTW9kdWxlKToKICAgICIiIkNoZWFwIGRlcHRod2lzZSBkaWxhdGVkIDN4MyBjb252ICsgQk4gKyBhY3RpdmF0aW9uLiBE'
        'ZXB0aHdpc2Uga2VlcHMgdGhpcwogICAgYWZmb3JkYWJsZSBhdCBQMydzIGhpZ2ggc3BhdGlhbCByZXNvbHV0aW9uOyB0aGUgZGlsYXRpb24gcm'
        'F0ZSBjb250cm9scyBob3cKICAgIG11Y2ggbG9jYWwgY29udGV4dCBlYWNoIGJyYW5jaCBvZiBEUk0gZ2F0aGVycy4iIiIKICAgIGRlZiBfX2lu'
        'aXRfXyhzZWxmLCBjaGFubmVscywgZGlsYXRpb24pOgogICAgICAgIHN1cGVyKERlcHRod2lzZURpbGF0ZWRDb252LCBzZWxmKS5fX2luaXRfXy'
        'gpCiAgICAgICAgc2VsZi5kdyAgPSBubi5Db252MmQoY2hhbm5lbHMsIGNoYW5uZWxzLCBrZXJuZWxfc2l6ZT0zLCBzdHJpZGU9MSwKICAgICAg'
        'ICAgICAgICAgICAgICAgICAgICAgICAgcGFkZGluZz1kaWxhdGlvbiwgZGlsYXRpb249ZGlsYXRpb24sIGdyb3Vwcz1jaGFubmVscywgYmlhcz'
        '1GYWxzZSkKICAgICAgICBzZWxmLmJuICA9IG5uLkJhdGNoTm9ybTJkKGNoYW5uZWxzLCBlcHM9MC4wMDEsIG1vbWVudHVtPTAuMDMpCiAgICAg'
        'ICAgc2VsZi5hY3QgPSBubi5MZWFreVJlTFUoMC4xLCBpbnBsYWNlPVRydWUpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgcm'
        'V0dXJuIHNlbGYuYWN0KHNlbGYuYm4oc2VsZi5kdyh4KSkpCgoKY2xhc3MgRFJNKG5uLk1vZHVsZSk6CiAgICAiIiJEZXRhaWwgUmVjb3Zlcnkg'
        'TW9kdWxlLgoKICAgIFJERk5ldCdzIFJGRSBtb2R1bGUgZW5sYXJnZXMgdGhlIHJlY2VwdGl2ZSBmaWVsZCBvbmx5IGF0IHRoZSBjb2Fyc2UgUD'
        'UvU1BQCiAgICBzdGFnZS4gU21hbGwvdGhpbiBvYmplY3RzIChlLmcuIGJpY3ljbGUgLS0gdGhlIHdlYWtlc3QgY2xhc3Mgb24gYm90aAogICAg'
        'Vk9DLUZPRyBhbmQgUlRUUyBpbiB0aGUgYmFzZSBwYXBlcidzIG93biBwZXItY2xhc3MgcmVzdWx0cykgYXJlIGRldGVjdGVkCiAgICBhdCB0aG'
        'UgZmluZXIgUDMgc2NhbGUsIHdoaWNoIHJlY2VpdmVzIG5vIGVxdWl2YWxlbnQgZGV0YWlsLXJlY292ZXJ5CiAgICB0cmVhdG1lbnQsIGFuZCBm'
        'b2cgc3BlY2lmaWNhbGx5IGF0dGVudWF0ZXMgdGhlIGhpZ2gtZnJlcXVlbmN5IGRldGFpbCBzdWNoCiAgICBvYmplY3RzIGRlcGVuZCBvbi4gRF'
        'JNIGlzIGEgbGlnaHR3ZWlnaHQsIHJlc2lkdWFsIHNpYmxpbmcgb2YgUkZFIGFwcGxpZWQKICAgIGF0IFAzOiB0d28gZGVwdGh3aXNlIGRpbGF0'
        'ZWQtY29udiBicmFuY2hlcyAoZGlsYXRpb24gMSBhbmQgMikgcmVjb3ZlcgogICAgbG9jYWwgZGV0YWlsIGF0IGxvdyBjaGFubmVsIHdpZHRoLC'
        'BhbiBFQ0EgYmxvY2sgcmVjYWxpYnJhdGVzIGNoYW5uZWxzLAogICAgYW5kIGEgMXgxIHByb2plY3Rpb24gZnVzZXMgYmFjayB0byB0aGUgb3Jp'
        'Z2luYWwgY2hhbm5lbCBjb3VudC4KCiAgICBJbXBsZW1lbnRlZCBhcyBhIHplcm8taW5pdGlhbGl6ZWQgcmVzaWR1YWwgYnJhbmNoIC0tCiAgIC'
        'AgICAgUDNfb3V0ID0gUDMgKyBmKFAzKQogICAgLS0gd2l0aCBmJ3MgZmluYWwgcHJvamVjdGlvbiB3ZWlnaHQgYW5kIGJpYXMgaW5pdGlhbGl6'
        'ZWQgdG8gemVybywgc28gdGhlCiAgICBtb2R1bGUgaXMgYW4gZXhhY3QgaWRlbnRpdHkgZnVuY3Rpb24gYXQgdGhlIHN0YXJ0IG9mIGZpbmUtdH'
        'VuaW5nIGZyb20gYQogICAgY29udmVyZ2VkIGNoZWNrcG9pbnQuIEl0IGNhbiBvbmx5IGJlZ2luIGNvbnRyaWJ1dGluZyBhcyB0cmFpbmluZyBt'
        'b3ZlcwogICAgdGhvc2Ugd2VpZ2h0cyBhd2F5IGZyb20gemVybywgd2hpY2ggYm91bmRzIHRoZSBkb3duc2lkZSBpZiB0aGUgZmluZS10dW5lCi'
        'AgICBidWRnZXQgdHVybnMgb3V0IHRvIGJlIHRvbyBzaG9ydCBmb3IgaXQgdG8gbGVhcm4gc29tZXRoaW5nIHVzZWZ1bC4KICAgICIiIgogICAg'
        'ZGVmIF9faW5pdF9fKHNlbGYsIGNoYW5uZWxzLCByZWR1Y3Rpb249Mik6CiAgICAgICAgc3VwZXIoRFJNLCBzZWxmKS5fX2luaXRfXygpCiAgIC'
        'AgICAgaGlkZGVuID0gbWF4KGNoYW5uZWxzIC8vIHJlZHVjdGlvbiwgOCkKICAgICAgICBzZWxmLnJlZHVjZSAgPSBDb252KGNoYW5uZWxzLCBo'
        'aWRkZW4sIDEsIDEpCiAgICAgICAgc2VsZi5icmFuY2gxID0gRGVwdGh3aXNlRGlsYXRlZENvbnYoaGlkZGVuLCBkaWxhdGlvbj0xKQogICAgIC'
        'AgIHNlbGYuYnJhbmNoMiA9IERlcHRod2lzZURpbGF0ZWRDb252KGhpZGRlbiwgZGlsYXRpb249MikKICAgICAgICBzZWxmLmVjYSAgICAgPSBF'
        'Q0EoaGlkZGVuICogMikKICAgICAgICBzZWxmLnByb2plY3QgPSBubi5Db252MmQoaGlkZGVuICogMiwgY2hhbm5lbHMsIGtlcm5lbF9zaXplPT'
        'EsIHN0cmlkZT0xLCBiaWFzPVRydWUpCgogICAgICAgICMgWmVyby1pbml0OiB0aGlzIGJyYW5jaCBtdXN0IHN0YXJ0IGFzIGFuIGlkZW50aXR5'
        'IGZ1bmN0aW9uIHNvIGl0CiAgICAgICAgIyBjYW5ub3QgZGVzdGFiaWxpemUgYW4gYWxyZWFkeS1jb252ZXJnZWQgY2hlY2twb2ludCBvbiBlcG'
        '9jaCAwLgogICAgICAgIG5uLmluaXQuemVyb3NfKHNlbGYucHJvamVjdC53ZWlnaHQpCiAgICAgICAgbm4uaW5pdC56ZXJvc18oc2VsZi5wcm9q'
        'ZWN0LmJpYXMpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgeSAgPSBzZWxmLnJlZHVjZSh4KQogICAgICAgIHkxID0gc2VsZi'
        '5icmFuY2gxKHkpCiAgICAgICAgeTIgPSBzZWxmLmJyYW5jaDIoeSkKICAgICAgICB5ICA9IHRvcmNoLmNhdChbeTEsIHkyXSwgZGltPTEpCiAg'
        'ICAgICAgeSAgPSBzZWxmLmVjYSh5KQogICAgICAgIHkgID0gc2VsZi5wcm9qZWN0KHkpCiAgICAgICAgcmV0dXJuIHggKyB5CgoKZGVmIHByaW'
        '50X21vZGVsX2Zsb3BzX2FuZF9wYXJhbXMobW9kZWwsIGlucHV0cyk6CiAgICBmbG9wcywgcGFyYW1zID0gcHJvZmlsZShtb2RlbCwgaW5wdXRz'
        'PWlucHV0cykKICAgIHByaW50KGYiRkxPUHM6IHtmbG9wcyAvIDFlOTouMmZ9IEdGTE9QcyIpCiAgICBwcmludChmIlBhcmFtZXRlcnM6IHtwYX'
        'JhbXMgLyAxZTY6LjJmfSBNIikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgZmVhdDEgPSB0b3JjaC5yYW5kbigxLCA2NCwgMTYw'
        'LCAxNjApCiAgICBmZWF0MiA9IHRvcmNoLnJhbmRuKDEsIDEyOCwgODAsIDgwKSAKICAgIGZlYXQzID0gdG9yY2gucmFuZG4oMSwgMjU2LCA0MC'
        'wgNDApICAKICAgIGVuY29kZXIgPSBMTUROZXQoKQogICAgcHJpbnRfbW9kZWxfZmxvcHNfYW5kX3BhcmFtcyhlbmNvZGVyLCAoZmVhdDEsIGZl'
        'YXQyLCBmZWF0MykpCiAgICAK'
    ),
    'nets/model.py': (
        'aW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgpmcm9tIG5ldHMuQ29tbW9uIGltcG9ydCBDb252LCBTUFBFTEFOLCBEUk0KZnJvbS'
        'BuZXRzLmJhY2tib25lIGltcG9ydCBCYWNrYm9uZSwgTXVsdGlfQ29uY2F0X0Jsb2NrCnRyeToKICAgIGZyb20gY29uZmlnIGltcG9ydCBVU0Vf'
        'RFJNCmV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgIFVTRV9EUk0gPSBGYWxzZQoKZGVmIGZ1c2VfY29udl9hbmRfYm4oY29udiwgYm4pOgogICAgZn'
        'VzZWRjb252ID0gbm4uQ29udjJkKGNvbnYuaW5fY2hhbm5lbHMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgY29udi5vdXRfY2hhbm5lbHMs'
        'CiAgICAgICAgICAgICAgICAgICAgICAgICAga2VybmVsX3NpemU9Y29udi5rZXJuZWxfc2l6ZSwKICAgICAgICAgICAgICAgICAgICAgICAgIC'
        'BzdHJpZGU9Y29udi5zdHJpZGUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgcGFkZGluZz1jb252LnBhZGRpbmcsCiAgICAgICAgICAgICAg'
        'ICAgICAgICAgICAgZ3JvdXBzPWNvbnYuZ3JvdXBzLAogICAgICAgICAgICAgICAgICAgICAgICAgIGJpYXM9VHJ1ZSkucmVxdWlyZXNfZ3JhZF'
        '8oRmFsc2UpLnRvKGNvbnYud2VpZ2h0LmRldmljZSkKCiAgICB3X2NvbnYgID0gY29udi53ZWlnaHQuY2xvbmUoKS52aWV3KGNvbnYub3V0X2No'
        'YW5uZWxzLCAtMSkKICAgIHdfYm4gICAgPSB0b3JjaC5kaWFnKGJuLndlaWdodC5kaXYodG9yY2guc3FydChibi5lcHMgKyBibi5ydW5uaW5nX3'
        'ZhcikpKQogICAgZnVzZWRjb252LndlaWdodC5jb3B5Xyh0b3JjaC5tbSh3X2JuLCB3X2NvbnYpLnZpZXcoZnVzZWRjb252LndlaWdodC5zaGFw'
        'ZSkuZGV0YWNoKCkpCgogICAgYl9jb252ICA9IHRvcmNoLnplcm9zKGNvbnYud2VpZ2h0LnNpemUoMCksIGRldmljZT1jb252LndlaWdodC5kZX'
        'ZpY2UpIGlmIGNvbnYuYmlhcyBpcyBOb25lIGVsc2UgY29udi5iaWFzCiAgICBiX2JuICAgID0gYm4uYmlhcyAtIGJuLndlaWdodC5tdWwoYm4u'
        'cnVubmluZ19tZWFuKS5kaXYodG9yY2guc3FydChibi5ydW5uaW5nX3ZhciArIGJuLmVwcykpCiAgICBmdXNlZGNvbnYuYmlhcy5jb3B5XygodG'
        '9yY2gubW0od19ibiwgYl9jb252LnJlc2hhcGUoLTEsIDEpKS5yZXNoYXBlKC0xKSArIGJfYm4pLmRldGFjaCgpKQogICAgcmV0dXJuIGZ1c2Vk'
        'Y29udgoKY2xhc3MgTVAobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBrPTIpOgogICAgICAgIHN1cGVyKE1QLCBzZWxmKS5fX2'
        'luaXRfXygpCiAgICAgICAgc2VsZi5tMSA9IG5uLk1heFBvb2wyZChrZXJuZWxfc2l6ZT1rLCBzdHJpZGU9aykKICAgICAgICBzZWxmLm0yID0g'
        'bm4uQXZnUG9vbDJkKGtlcm5lbF9zaXplPWssIHN0cmlkZT1rKQogICAgICAgIHNlbGYudXAgPSBubi5VcHNhbXBsZShzY2FsZV9mYWN0b3I9Mi'
        'kKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICB4MSA9IHNlbGYubTEoeCkKICAgICAgICB4MiA9IHNlbGYubTIoeCkKICAgICAg'
        'ICByZXR1cm4gc2VsZi51cCh4MSArIHgyKQogICAgCmNsYXNzIFlvbG9Cb2R5KG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgYW'
        '5jaG9yc19tYXNrLCBudW1fY2xhc3Nlcyk6CiAgICAgICAgc3VwZXIoWW9sb0JvZHksIHNlbGYpLl9faW5pdF9fKCkKICAgICAgICB0cmFuc2l0'
        'aW9uX2NoYW5uZWxzID0gMTYKICAgICAgICBibG9ja19jaGFubmVscyAgICAgID0gMTYKICAgICAgICBwYW5ldF9jaGFubmVscyAgICAgID0gMT'
        'YKICAgICAgICBlICAgICAgICAgICAgICAgICAgID0gMQogICAgICAgIG4gICAgICAgICAgICAgICAgICAgPSAyCiAgICAgICAgaWRzICAgICAg'
        'ICAgICAgICAgICA9IFstMSwgLTIsIC0zLCAtNF0KCiAgICAgICAgc2VsZi5iYWNrYm9uZSAgID0gQmFja2JvbmUodHJhbnNpdGlvbl9jaGFubm'
        'VscywgYmxvY2tfY2hhbm5lbHMsIG4pCgogICAgICAgIHNlbGYudXBzYW1wbGUgICA9IG5uLlVwc2FtcGxlKHNjYWxlX2ZhY3Rvcj0yLCBtb2Rl'
        'PSJuZWFyZXN0IikKCiAgICAgICAgc2VsZi5zcHBlbGFuICAgICAgICAgICAgICAgID0gU1BQRUxBTih0cmFuc2l0aW9uX2NoYW5uZWxzICogMz'
        'IsIHRyYW5zaXRpb25fY2hhbm5lbHMgKiAxNikKICAgICAgICBzZWxmLmNvbnZfZm9yX1A1ICAgICAgICAgICAgPSBDb252KHRyYW5zaXRpb25f'
        'Y2hhbm5lbHMgKiAxNiwgdHJhbnNpdGlvbl9jaGFubmVscyAqIDgpCiAgICAgICAgc2VsZi5jb252X2Zvcl9mZWF0MiAgICAgICAgID0gQ29udi'
        'h0cmFuc2l0aW9uX2NoYW5uZWxzICogMTYsIHRyYW5zaXRpb25fY2hhbm5lbHMgKiA4KQogICAgICAgIHNlbGYuY29udjNfZm9yX3Vwc2FtcGxl'
        'MSAgICA9IE11bHRpX0NvbmNhdF9CbG9jayh0cmFuc2l0aW9uX2NoYW5uZWxzICogMTYsIHBhbmV0X2NoYW5uZWxzICogNCwgdHJhbnNpdGlvbl'
        '9jaGFubmVscyAqIDgsIGU9ZSwgbj1uLCBpZHM9aWRzKQoKICAgICAgICBzZWxmLmNvbnZfZm9yX1A0ICAgICAgICAgICAgPSBDb252KHRyYW5z'
        'aXRpb25fY2hhbm5lbHMgKiA4LCB0cmFuc2l0aW9uX2NoYW5uZWxzICogNCkKICAgICAgICBzZWxmLmNvbnZfZm9yX2ZlYXQxICAgICAgICAgPS'
        'BDb252KHRyYW5zaXRpb25fY2hhbm5lbHMgKiA4LCB0cmFuc2l0aW9uX2NoYW5uZWxzICogNCkKICAgICAgICBzZWxmLmNvbnYzX2Zvcl91cHNh'
        'bXBsZTIgICAgPSBNdWx0aV9Db25jYXRfQmxvY2sodHJhbnNpdGlvbl9jaGFubmVscyAqIDgsIHBhbmV0X2NoYW5uZWxzICogMiwgdHJhbnNpdG'
        'lvbl9jaGFubmVscyAqIDQsIGU9ZSwgbj1uLCBpZHM9aWRzKQoKICAgICAgICAjIERldGFpbCBSZWNvdmVyeSBNb2R1bGU6IHplcm8taW5pdCBy'
        'ZXNpZHVhbCBkZXRhaWwvYXR0ZW50aW9uIGJsb2NrIGF0CiAgICAgICAgIyB0aGUgZmluZXN0IChQMykgc2NhbGUsIHRhcmdldGluZyB0aGUgc2'
        '1hbGwvdGhpbi1vYmplY3Qgd2Vha25lc3MKICAgICAgICAjIChiaWN5Y2xlLCBtb3RvcmJpa2UpIHRoYXQgUkRGTmV0J3MgUkZFIC0tIGFwcGxp'
        'ZWQgb25seSBhdCB0aGUgY29hcnNlCiAgICAgICAgIyBQNS9TUFAgc3RhZ2UgLS0gZG9lcyBub3QgYWRkcmVzcy4gRmFsbHMgYmFjayB0byBhIG'
        '5vLW9wIGlkZW50aXR5CiAgICAgICAgIyB3aGVuIGNvbmZpZy5VU0VfRFJNIGlzIEZhbHNlLCBzbyB0aGUgImNvbnRyb2wiIGFyY2hpdGVjdHVy'
        'ZSB1c2VkIHRvCiAgICAgICAgIyBpc29sYXRlIHRoaXMgbW9kdWxlJ3MgY29udHJpYnV0aW9uIGlzIGJ5dGUtZm9yLWJ5dGUgdGhlIGJhc2UgcG'
        'FwZXIncy4KICAgICAgICBzZWxmLmRybSA9IERSTSh0cmFuc2l0aW9uX2NoYW5uZWxzICogNCkgaWYgVVNFX0RSTSBlbHNlIG5uLklkZW50aXR5'
        'KCkKCiAgICAgICAgc2VsZi5kb3duX3NhbXBsZTEgICAgICAgICAgID0gQ29udih0cmFuc2l0aW9uX2NoYW5uZWxzICogNCwgdHJhbnNpdGlvbl'
        '9jaGFubmVscyAqIDgsIGs9Mywgcz0yKQogICAgICAgIHNlbGYuY29udjNfZm9yX2Rvd25zYW1wbGUxICA9IE11bHRpX0NvbmNhdF9CbG9jayh0'
        'cmFuc2l0aW9uX2NoYW5uZWxzICogMTYsIHBhbmV0X2NoYW5uZWxzICogNCwgdHJhbnNpdGlvbl9jaGFubmVscyAqIDgsIGU9ZSwgbj1uLCBpZH'
        'M9aWRzKQoKICAgICAgICBzZWxmLmRvd25fc2FtcGxlMiAgICAgICAgICAgPSBDb252KHRyYW5zaXRpb25fY2hhbm5lbHMgKiA4LCB0cmFuc2l0'
        'aW9uX2NoYW5uZWxzICogMTYsIGs9Mywgcz0yKQogICAgICAgIHNlbGYuY29udjNfZm9yX2Rvd25zYW1wbGUyICA9IE11bHRpX0NvbmNhdF9CbG'
        '9jayh0cmFuc2l0aW9uX2NoYW5uZWxzICogMzIsIHBhbmV0X2NoYW5uZWxzICogOCwgdHJhbnNpdGlvbl9jaGFubmVscyAqIDE2LCBlPWUsIG49'
        'biwgaWRzPWlkcykKCiAgICAgICAgc2VsZi5wZiA9IE1QKCkKCiAgICAgICAgc2VsZi5yZXBfY29udl8xID0gQ29udih0cmFuc2l0aW9uX2NoYW'
        '5uZWxzICogNCwgdHJhbnNpdGlvbl9jaGFubmVscyAqIDgsIDMsIDEpCiAgICAgICAgc2VsZi5yZXBfY29udl8yID0gQ29udih0cmFuc2l0aW9u'
        'X2NoYW5uZWxzICogOCwgdHJhbnNpdGlvbl9jaGFubmVscyAqIDE2LCAzLCAxKQogICAgICAgIHNlbGYucmVwX2NvbnZfMyA9IENvbnYodHJhbn'
        'NpdGlvbl9jaGFubmVscyAqIDE2LCB0cmFuc2l0aW9uX2NoYW5uZWxzICogMzIsIDMsIDEpCgogICAgICAgIHNlbGYueW9sb19oZWFkX1AzID0g'
        'bm4uQ29udjJkKHRyYW5zaXRpb25fY2hhbm5lbHMgKiA4LCBsZW4oYW5jaG9yc19tYXNrWzJdKSAqICg1ICsgbnVtX2NsYXNzZXMpLCAxKQogIC'
        'AgICAgIHNlbGYueW9sb19oZWFkX1A0ID0gbm4uQ29udjJkKHRyYW5zaXRpb25fY2hhbm5lbHMgKiAxNiwgbGVuKGFuY2hvcnNfbWFza1sxXSkg'
        'KiAoNSArIG51bV9jbGFzc2VzKSwgMSkKICAgICAgICBzZWxmLnlvbG9faGVhZF9QNSA9IG5uLkNvbnYyZCh0cmFuc2l0aW9uX2NoYW5uZWxzIC'
        'ogMzIsIGxlbihhbmNob3JzX21hc2tbMF0pICogKDUgKyBudW1fY2xhc3NlcyksIDEpCgogICAgZGVmIGZ1c2Uoc2VsZik6CiAgICAgICAgcHJp'
        'bnQoJ0Z1c2luZyBsYXllcnMuLi4gJykKICAgICAgICBmb3IgbSBpbiBzZWxmLm1vZHVsZXMoKToKICAgICAgICAgICAgaWYgdHlwZShtKSBpcy'
        'BDb252IGFuZCBoYXNhdHRyKG0sICdibicpOgogICAgICAgICAgICAgICAgbS5jb252ID0gZnVzZV9jb252X2FuZF9ibihtLmNvbnYsIG0uYm4p'
        'CiAgICAgICAgICAgICAgICBkZWxhdHRyKG0sICdibicpCiAgICAgICAgICAgICAgICBtLmZvcndhcmQgPSBtLmZ1c2Vmb3J3YXJkCiAgICAgIC'
        'AgcmV0dXJuIHNlbGYKICAgIAogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgaWYgc2VsZi50cmFpbmluZzoKICAgICAgICAgICAg'
        'ZmVhdDEsIGZlYXQyLCBmZWF0MywgZGVoYXppbmcgPSBzZWxmLmJhY2tib25lLmZvcndhcmQoeCkKICAgICAgICBlbHNlOgogICAgICAgICAgIC'
        'BmZWF0MSwgZmVhdDIsIGZlYXQzID0gc2VsZi5iYWNrYm9uZS5mb3J3YXJkKHgpCgogICAgICAgIFA1ICAgICAgICAgID0gc2VsZi5zcHBlbGFu'
        'KGZlYXQzKQogICAgICAgIFA1X2NvbnYgICAgID0gc2VsZi5jb252X2Zvcl9QNShQNSkKICAgICAgICBQNV91cHNhbXBsZSA9IHNlbGYudXBzYW'
        '1wbGUoUDVfY29udikKICAgICAgICBQNCAgICAgICAgICA9IHRvcmNoLmNhdChbc2VsZi5jb252X2Zvcl9mZWF0MihmZWF0MiksIFA1X3Vwc2Ft'
        'cGxlXSwgMSkKICAgICAgICBQNCAgICAgICAgICA9IHNlbGYuY29udjNfZm9yX3Vwc2FtcGxlMShQNCkKCiAgICAgICAgUDRfY29udiAgICAgPS'
        'BzZWxmLmNvbnZfZm9yX1A0KFA0KQogICAgICAgIFA0X3Vwc2FtcGxlID0gc2VsZi51cHNhbXBsZShQNF9jb252KQogICAgICAgIFAzICAgICAg'
        'ICAgID0gdG9yY2guY2F0KFtzZWxmLmNvbnZfZm9yX2ZlYXQxKGZlYXQxKSwgUDRfdXBzYW1wbGVdLCAxKQogICAgICAgIFAzICAgICAgICAgID'
        '0gc2VsZi5jb252M19mb3JfdXBzYW1wbGUyKFAzKQogICAgICAgIFAzICAgICAgICAgID0gc2VsZi5kcm0oUDMpCgogICAgICAgIFAzX2Rvd25z'
        'YW1wbGUgPSBzZWxmLmRvd25fc2FtcGxlMShQMykKICAgICAgICBQNCA9IHRvcmNoLmNhdChbUDNfZG93bnNhbXBsZSwgUDRdLCAxKQogICAgIC'
        'AgIFA0ID0gc2VsZi5jb252M19mb3JfZG93bnNhbXBsZTEoUDQpCiAgICAgICAgUDQgPSBzZWxmLnBmKFA0KQoKICAgICAgICBQNF9kb3duc2Ft'
        'cGxlID0gc2VsZi5kb3duX3NhbXBsZTIoUDQpCiAgICAgICAgUDUgPSB0b3JjaC5jYXQoW1A0X2Rvd25zYW1wbGUsIFA1XSwgMSkKICAgICAgIC'
        'BQNSA9IHNlbGYuY29udjNfZm9yX2Rvd25zYW1wbGUyKFA1KQogICAgICAgIAogICAgICAgIFAzID0gc2VsZi5yZXBfY29udl8xKFAzKQogICAg'
        'ICAgIFA0ID0gc2VsZi5yZXBfY29udl8yKFA0KQogICAgICAgIFA1ID0gc2VsZi5yZXBfY29udl8zKFA1KQoKICAgICAgICBvdXQyID0gc2VsZi'
        '55b2xvX2hlYWRfUDMoUDMpCiAgICAgICAgb3V0MSA9IHNlbGYueW9sb19oZWFkX1A0KFA0KQogICAgICAgIG91dDAgPSBzZWxmLnlvbG9faGVh'
        'ZF9QNShQNSkKCiAgICAgICAgaWYgc2VsZi50cmFpbmluZzoKICAgICAgICAgICAgcmV0dXJuIFtvdXQwLCBvdXQxLCBvdXQyLCBkZWhhemluZ1'
        '0KICAgICAgICBlbHNlOgogICAgICAgICAgICByZXR1cm4gW291dDAsIG91dDEsIG91dDJd'
    ),
    'config.py': (
        'Q3VkYSAgICAgICAgICAgID0gVHJ1ZQpzZWVkICAgICAgICAgICAgPSAxMTQ1MTQKZGlzdHJpYnV0ZWQgICAgID0gRmFsc2UKc3luY19ibiAgIC'
        'AgICAgID0gRmFsc2UKZnAxNiAgICAgICAgICAgID0gRmFsc2UKY2xhc3Nlc19wYXRoICAgID0gJ21vZGVsX2RhdGEvcnR0c19jbGFzc2VzLnR4'
        'dCcKYW5jaG9yc19wYXRoICAgID0gJ21vZGVsX2RhdGEveW9sb19hbmNob3JzLnR4dCcKYW5jaG9yc19tYXNrICAgID0gW1s2LCA3LCA4XSwgWz'
        'MsIDQsIDVdLCBbMCwgMSwgMl1dCm1vZGVsX3BhdGggICAgICA9ICdtb2RlbF9kYXRhL3lvbG92N190aW55X3dlaWdodHMucHRoJwppbnB1dF9z'
        'aGFwZSAgICAgPSBbNjQwLCA2NDBdCnByZXRyYWluZWQgICAgICA9IEZhbHNlCkluaXRfRXBvY2ggICAgICAgICAgPSAwCkZyZWV6ZV9FcG9jaC'
        'AgICAgICAgPSAxMDAKRnJlZXplX2JhdGNoX3NpemUgICA9IDE2ClVuRnJlZXplX0Vwb2NoICAgICAgPSAzMDAKVW5mcmVlemVfYmF0Y2hfc2l6'
        'ZSA9IDE2CkZyZWV6ZV9UcmFpbiAgICAgICAgPSBUcnVlCkluaXRfbHIgICAgICAgICAgICAgPSAxZS0yCk1pbl9sciAgICAgICAgICAgICAgPS'
        'BJbml0X2xyICogMC4wMQpvcHRpbWl6ZXJfdHlwZSAgICAgID0gInNnZCIKbW9tZW50dW0gICAgICAgICAgICA9IDAuOTM3CndlaWdodF9kZWNh'
        'eSAgICAgICAgPSA1ZS00CmxyX2RlY2F5X3R5cGUgICAgICAgPSAiY29zIgpzYXZlX3BlcmlvZCAgICAgICAgID0gMTAKc2F2ZV9kaXIgICAgIC'
        'AgICAgICA9ICdsb2dzJwpldmFsX2ZsYWcgICAgICAgICAgID0gVHJ1ZQpldmFsX3BlcmlvZCAgICAgICAgID0gMTAKbnVtX3dvcmtlcnMgICAg'
        'ICAgICA9IDAKdHJhaW5fYW5ub3RhdGlvbl9wYXRoICAgPSAnMjAwN190cmFpbl9mb2cudHh0Jwp2YWxfYW5ub3RhdGlvbl9wYXRoICAgICA9IC'
        'cyMDA3X3ZhbF9mb2cudHh0JwpjbGVhcl9hbm5vdGF0aW9uX3BhdGggPSAnMjAwN190cmFpbi50eHQnCgojIC0tLSBUaGVzaXMgbW9kaWZpY2F0'
        'aW9ucyBvdmVyIHRoZSBiYXNlIHBhcGVyLCBhbGwgZGVmYXVsdCBPRkYgc28gdGhlIC0tLQojIC0tLSBjb21taXR0ZWQgcmVwbyByZXByb2R1Y2'
        'VzIHRoZSBiYXNlIHBhcGVyJ3MgZXhhY3QgYmVoYXZpb3IgdW5sZXNzICAgLS0tCiMgLS0tIGV4cGxpY2l0bHkgZW5hYmxlZCAoZS5nLiBieSB0'
        'aGUgZmluZS10dW5pbmcgbm90ZWJvb2spLiAgICAgICAgICAgICAtLS0KIyBEUk06IHplcm8taW5pdCByZXNpZHVhbCBEZXRhaWwgUmVjb3Zlcn'
        'kgTW9kdWxlIGF0IHRoZSBQMyAoZmluZXN0KSBzY2FsZSwKIyAgICAgIHRhcmdldGluZyB0aGUgc21hbGwvdGhpbi1vYmplY3Qgd2Vha25lc3Mg'
        'KGJpY3ljbGUsIG1vdG9yYmlrZSkuClVTRV9EUk0gICAgICAgICA9IEZhbHNlCiMgV2lzZS1Jb1U6IGR5bmFtaWMgbm9uLW1vbm90b25pYyBmb2'
        'N1c2luZyBhcHBsaWVkIG9uIHRvcCBvZiB0aGUgZXhpc3RpbmcKIyAgICAgICAgICAgQ0lvVSBib3gtcmVncmVzc2lvbiBsb3NzLgpVU0VfV0lT'
        'RV9JT1UgICAgPSBGYWxzZQojIERXQTogRHluYW1pYyBXZWlnaHQgQXZlcmFnaW5nIChMaXUgZXQgYWwuLCBDVlBSIDIwMTkpIGZvciB0aGUgZG'
        'V0ZWN0aW9uIC8KIyAgICAgIGRlaGF6ZSB0YXNrIGJhbGFuY2UsIHJlcGxhY2luZyB0aGUgZml4ZWQgbGFtYmRhPTAuMSBiZWxvdy4KVVNFX0RX'
        'QSAgICAgICAgID0gRmFsc2U='
    ),
    'nets/yolo_training.py': (
        '...'
        'PSBscgo='
    ),
    'utils/utils_fit.py': (
        'aW1wb3J0IG9zCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KZnJvbSB0cWRtIGltcG9ydCB0cWRtCmZyb20gdXRpbHMudXRpbH'
        'MgaW1wb3J0IGdldF9scgoKZGVmIGZpdF9vbmVfZXBvY2gobW9kZWxfdHJhaW4sIG1vZGVsLCBlbWEsIHlvbG9fbG9zcywgbG9zc19oaXN0b3J5'
        'LCBldmFsX2NhbGxiYWNrLCBvcHRpbWl6ZXIsIGVwb2NoLCBlcG9jaF9zdGVwLCBnZW4sIEVwb2NoLCBjdWRhLCBmcDE2LCBzY2FsZXIsIHNhdm'
        'VfcGVyaW9kLCBzYXZlX2RpciwgbG9jYWxfcmFuaz0wKToKICAgIGxvc3MgICAgICAgID0gMAogICAgRGVoYXp5X2xvc3MgPSAwCiAgICBEZXRf'
        'bG9zcyAgICA9IDAKICAgIGNyaXRlcmlvbiA9IG5uLk1TRUxvc3MoKQoKICAgICMgRHluYW1pYyBXZWlnaHQgQXZlcmFnaW5nIChEV0EpIHRhc2'
        'sgd2VpZ2h0cywgcmVjb21wdXRlZCBvbmNlIHBlciBlcG9jaAogICAgIyBmcm9tIHRoZSBwcmV2aW91cyB0d28gZXBvY2hzJyBhdmVyYWdlIGxv'
        'c3NlcyAoZXF1YWwgd2VpZ2h0cyB1bnRpbCB0aGVuLAogICAgIyBvciBhbHdheXMgZXF1YWwvZml4ZWQtbGFtYmRhLWNvbXBhdGlibGUgaWYgY2'
        '9uZmlnLlVTRV9EV0EgaXMgRmFsc2UpLgogICAgd19kZXQsIHdfZGVoYXplID0geW9sb19sb3NzLmdldF90YXNrX3dlaWdodHMoKQoKICAgIGlm'
        'IGxvY2FsX3JhbmsgPT0gMDoKICAgICAgICBwcmludCgnU3RhcnQgVHJhaW4nKQogICAgICAgIHByaW50KCdUYXNrIHdlaWdodHMgdGhpcyBlcG'
        '9jaCAtPiBkZXRlY3Rpb246ICUuM2YsIGRlaGF6ZTogJS4zZicgJSAod19kZXQsIHdfZGVoYXplKSkKICAgICAgICBwYmFyID0gdHFkbSh0b3Rh'
        'bD1lcG9jaF9zdGVwLGRlc2M9ZidFcG9jaCB7ZXBvY2ggKyAxfS97RXBvY2h9Jyxwb3N0Zml4PWRpY3QsbWluaW50ZXJ2YWw9MC4zKQogICAgIC'
        'AgIG1vZGVsX3RyYWluLnRyYWluKCkKCiAgICBmb3IgaXRlcmF0aW9uLCBiYXRjaCBpbiBlbnVtZXJhdGUoZ2VuKToKICAgICAgICBpZiBpdGVy'
        'YXRpb24gPj0gZXBvY2hfc3RlcDoKICAgICAgICAgICAgYnJlYWsKICAgICAgICBpbWFnZXMsIHRhcmdldHMsIGNsZWFuID0gYmF0Y2hbMF0sIG'
        'JhdGNoWzFdLCBiYXRjaFsyXQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBpZiBjdWRhOgogICAgICAgICAgICAg'
        'ICAgaW1hZ2VzICA9IGltYWdlcy5jdWRhKGxvY2FsX3JhbmspCiAgICAgICAgICAgICAgICB0YXJnZXRzID0gdGFyZ2V0cy5jdWRhKGxvY2FsX3'
        'JhbmspCiAgICAgICAgICAgICAgICBjbGVhbiA9IGNsZWFuLmN1ZGEobG9jYWxfcmFuaykKICAgICAgICAgICAgICAgIGhhenlfYW5kX2NsZWFy'
        'ID0gdG9yY2guY2F0KFtpbWFnZXMsIGNsZWFuXSwgZGltID0gMCkuY3VkYSgpCiAgICAgICAgb3B0aW1pemVyLnplcm9fZ3JhZCgpCgogICAgIC'
        'AgIGlmIG5vdCBmcDE2OgogICAgICAgICAgICBvdXRwdXRzICAgICAgICAgPSBtb2RlbF90cmFpbihoYXp5X2FuZF9jbGVhcikKICAgICAgICAg'
        'ICAgZGV0ZWN0X291dHB1dHMgPSBbb3V0cHV0c1swXSxvdXRwdXRzWzFdLG91dHB1dHNbMl1dCiAgICAgICAgICAgIGxvc3NfZGV0ZWN0aW9uIC'
        'AgICAgPSB5b2xvX2xvc3MoZGV0ZWN0X291dHB1dHMsIHRhcmdldHMsIGltYWdlcykKICAgICAgICAgICAgbG9zc19kZWhhenkgICAgID0gY3Jp'
        'dGVyaW9uKG91dHB1dHNbM10sIGNsZWFuKQogICAgICAgICAgICBsb3NzX3ZhbHVlICAgICAgPSB3X2RldCAqIGxvc3NfZGV0ZWN0aW9uICsgMC'
        '4xICogd19kZWhhemUgKiBsb3NzX2RlaGF6eQogICAgICAgICAgICBsb3NzX3ZhbHVlLmJhY2t3YXJkKCkKICAgICAgICAgICAgb3B0aW1pemVy'
        'LnN0ZXAoKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGZyb20gdG9yY2guY3VkYS5hbXAgaW1wb3J0IGF1dG9jYXN0CiAgICAgICAgICAgIH'
        'dpdGggYXV0b2Nhc3QoKToKICAgICAgICAgICAgICAgIG91dHB1dHMgICAgICAgICA9IG1vZGVsX3RyYWluKGltYWdlcykKICAgICAgICAgICAg'
        'ICAgIGxvc3NfdmFsdWUgICAgICA9IHlvbG9fbG9zcyhvdXRwdXRzLCB0YXJnZXRzLCBpbWFnZXMpCiAgICAgICAgICAgIHNjYWxlci5zY2FsZS'
        'hsb3NzX3ZhbHVlKS5iYWNrd2FyZCgpCiAgICAgICAgICAgIHNjYWxlci5zdGVwKG9wdGltaXplcikKICAgICAgICAgICAgc2NhbGVyLnVwZGF0'
        'ZSgpCiAgICAgICAgaWYgZW1hOgogICAgICAgICAgICBlbWEudXBkYXRlKG1vZGVsX3RyYWluKQogICAgICAgIERlaGF6eV9sb3NzICs9IGxvc3'
        'NfZGVoYXp5Lml0ZW0oKQogICAgICAgIERldF9sb3NzICAgICs9IGxvc3NfZGV0ZWN0aW9uLml0ZW0oKQogICAgICAgIGxvc3MgICAgICAgICs9'
        'IGxvc3NfdmFsdWUuaXRlbSgpCiAgICAgICAgaWYgbG9jYWxfcmFuayA9PSAwOgogICAgICAgICAgICBwYmFyLnNldF9wb3N0Zml4KCoqeydsb3'
        'NzJyAgOiBsb3NzIC8gKGl0ZXJhdGlvbiArIDEpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICdsb3NzX2RldGVjdGlvbicgIDog'
        'RGV0X2xvc3MgLyAoaXRlcmF0aW9uICsgMSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgJ0RlaGF6eV9sb3NzJzogRGVoYXp5X2'
        'xvc3MgLyAoaXRlcmF0aW9uICsgMSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgJ2xyJyAgICA6IGdldF9scihvcHRpbWl6ZXIp'
        'fSkKICAgICAgICAgICAgcGJhci51cGRhdGUoMSkKCiAgICAjIEZlZWQgdGhpcyBlcG9jaCdzIHJhdyAodW53ZWlnaHRlZCkgdGFzayBsb3NzZX'
        'MgYmFjayBpbnRvIHRoZSBEV0EKICAgICMgaGlzdG9yeSBzbyBuZXh0IGVwb2NoJ3Mgd2VpZ2h0cyBjYW4gYmUgY29tcHV0ZWQuIEhhcm1sZXNz'
        'IG5vLW9wIHdoZW4KICAgICMgY29uZmlnLlVTRV9EV0EgaXMgRmFsc2UuCiAgICB5b2xvX2xvc3MudXBkYXRlX3Rhc2tfbG9zc19oaXN0b3J5KE'
        'RldF9sb3NzIC8gZXBvY2hfc3RlcCwgRGVoYXp5X2xvc3MgLyBlcG9jaF9zdGVwKQoKICAgIGlmIGVtYToKICAgICAgICBtb2RlbF90cmFpbl9l'
        'dmFsID0gZW1hLmVtYQogICAgZWxzZToKICAgICAgICBtb2RlbF90cmFpbl9ldmFsID0gbW9kZWxfdHJhaW4uZXZhbCgpCgogICAgaWYgbG9jYW'
        'xfcmFuayA9PSAwOgogICAgICAgIHBiYXIuY2xvc2UoKQogICAgICAgIGxvc3NfaGlzdG9yeS5hcHBlbmRfbG9zcyhlcG9jaCArIDEsIGxvc3Mg'
        'LyBlcG9jaF9zdGVwKQogICAgICAgIGV2YWxfY2FsbGJhY2sub25fZXBvY2hfZW5kKGVwb2NoICsgMSwgbW9kZWxfdHJhaW5fZXZhbCkKICAgIC'
        'AgICBwcmludCgnRXBvY2g6Jysgc3RyKGVwb2NoICsgMSkgKyAnLycgKyBzdHIoRXBvY2gpKQogICAgICAgIHByaW50KCdUb3RhbCBMb3NzOiAl'
        'LjNmJyAlIChsb3NzIC8gZXBvY2hfc3RlcCkpCiAgICAgICAgaWYgZW1hOgogICAgICAgICAgICBzYXZlX3N0YXRlX2RpY3QgPSBlbWEuZW1hLn'
        'N0YXRlX2RpY3QoKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNhdmVfc3RhdGVfZGljdCA9IG1vZGVsLnN0YXRlX2RpY3QoKQogICAgICAg'
        'IGlmIChlcG9jaCArIDEpICUgc2F2ZV9wZXJpb2QgPT0gMCBvciBlcG9jaCArIDEgPT0gRXBvY2g6CiAgICAgICAgICAgIHRvcmNoLnNhdmUoc2'
        'F2ZV9zdGF0ZV9kaWN0LCBvcy5wYXRoLmpvaW4oc2F2ZV9kaXIsICJlcCUwM2QtbG9zcyUuM2YucHRoIiAlIChlcG9jaCArIDEsIGxvc3MgLyBl'
        'cG9jaF9zdGVwKSkpCgogICAgICAgICMgQmVzdC1jaGVja3BvaW50IHNlbGVjdGlvbjogcHJlZmVyIHZhbGlkYXRpb24gbUFQICh3aGF0IHdlIG'
        'FjdHVhbGx5CiAgICAgICAgIyBjYXJlIGFib3V0KSBvbmNlIGl0J3MgYXZhaWxhYmxlLCBzaW5jZSB0cmFpbiBsb3NzIHN0b3BzIGJlaW5nIGEK'
        'ICAgICAgICAjIHJlbGlhYmxlIHByb3h5IGZvciBhY2N1cmFjeSB3aGVuIERXQSBpcyByZXdlaWdodGluZyB0aGUgbG9zcwogICAgICAgICMgZm'
        '9ybXVsYXRpb24gZXBvY2ggdG8gZXBvY2guIEZhbGxzIGJhY2sgdG8gdHJhaW4gbG9zcyBvbmx5IG9uIHRoZQogICAgICAgICMgZXBvY2hzIGJl'
        'Zm9yZSB0aGUgZmlyc3QgbUFQIGV2YWx1YXRpb24gaGFzIHJ1bi4KICAgICAgICByZWFsX21hcHMgICAgICAgPSBldmFsX2NhbGxiYWNrLm1hcH'
        'NbMTpdCiAgICAgICAganVzdF9ldmFsdWF0ZWQgID0gbGVuKHJlYWxfbWFwcykgPiAwIGFuZCBldmFsX2NhbGxiYWNrLmVwb2NoZXNbLTFdID09'
        'IGVwb2NoICsgMQogICAgICAgIGlmIGp1c3RfZXZhbHVhdGVkIGFuZCBldmFsX2NhbGxiYWNrLm1hcHNbLTFdID49IG1heChyZWFsX21hcHMpOg'
        'ogICAgICAgICAgICAjIE5PVEU6IGV2YWxfY2FsbGJhY2subWFwcyBzdG9yZXMgbUFQIGFzIGEgMC0xIGZyYWN0aW9uLCBub3QgYQogICAgICAg'
        'ICAgICAjIHBlcmNlbnRhZ2UgLS0gbXVsdGlwbHkgYnkgMTAwIHNvIHRoaXMgcHJpbnQgaXNuJ3QgbWlzbGVhZGluZwogICAgICAgICAgICAjIC'
        'hwcmV2aW91c2x5IHByaW50ZWQgZS5nLiAiMC43NCUiIGZvciBhbiBhY3R1YWwgNzQuNCUgbUFQKS4KICAgICAgICAgICAgcHJpbnQoJ1NhdmUg'
        'YmVzdCBtb2RlbCB0byBiZXN0X2Vwb2NoX3dlaWdodHMucHRoICh2YWwgbUFQPSUuMmYlJSknICUgKGV2YWxfY2FsbGJhY2subWFwc1stMV0gKi'
        'AxMDApKQogICAgICAgICAgICB0b3JjaC5zYXZlKHNhdmVfc3RhdGVfZGljdCwgb3MucGF0aC5qb2luKHNhdmVfZGlyLCAiYmVzdF9lcG9jaF93'
        'ZWlnaHRzLnB0aCIpKQogICAgICAgIGVsaWYgbm90IHJlYWxfbWFwcyBhbmQgbG9zcyAvIGVwb2NoX3N0ZXAgPD0gbWluKGxvc3NfaGlzdG9yeS'
        '5sb3NzZXMpOgogICAgICAgICAgICBwcmludCgnU2F2ZSBiZXN0IG1vZGVsIHRvIGJlc3RfZXBvY2hfd2VpZ2h0cy5wdGggKHRyYWluIGxvc3Ms'
        'IG5vIG1BUCBldmFsIHlldCknKQogICAgICAgICAgICB0b3JjaC5zYXZlKHNhdmVfc3RhdGVfZGljdCwgb3MucGF0aC5qb2luKHNhdmVfZGlyLC'
        'AiYmVzdF9lcG9jaF93ZWlnaHRzLnB0aCIpKQogICAgICAgIHRvcmNoLnNhdmUoc2F2ZV9zdGF0ZV9kaWN0LCBvcy5wYXRoLmpvaW4oc2F2ZV9k'
        'aXIsICJsYXN0X2Vwb2NoX3dlaWdodHMucHRoIikpCgogICAgICAgICMgLS0tIFBlcmlvZGljIGV2YWx1YXRpb24gb24gdGhlIFJFQUwgVk9DLU'
        'ZPRy10ZXN0IC8gUlRUUy10ZXN0IHNldHMgLS0tCiAgICAgICAgIyBldmFsX2NhbGxiYWNrIGFib3ZlIG9ubHkgZXZlciBldmFsdWF0ZXMgVk9D'
        'LUZPRydzIG93biAqdmFsaWRhdGlvbioKICAgICAgICAjIHNwbGl0LiBFdmVyeSBSRUFMX0VWQUxfUEVSSU9EIGVwb2NocyAoYW5kIGF0IHRoZS'
        'BmaW5hbCBlcG9jaCkgd2UKICAgICAgICAjIGFkZGl0aW9uYWxseSBydW4gdGhlIGV4YWN0IHNhbWUgc3VicHJvY2Vzcy1pc29sYXRlZCBldmFs'
        'X29uZS5weSB1c2VkCiAgICAgICAgIyBmb3IgdGhlIGZpbmFsIGJhc2VsaW5lL2NvbnRyb2wvcHJvcG9zZWQgY29tcGFyaXNvbiBhZ2FpbnN0IH'
        'RoZSByZWFsCiAgICAgICAgIyBoZWxkLW91dCB0ZXN0IHNldHMsIGFuZCBkaWZmIHRoZSByZXN1bHQgYWdhaW5zdCB0aGUgYmFzZWxpbmUKICAg'
        'ICAgICAjIGNoZWNrcG9pbnQncyBudW1iZXJzIC0tIHNvIHJlZ3Jlc3Npb25zIGFyZSB2aXNpYmxlIGR1cmluZyB0cmFpbmluZywKICAgICAgIC'
        'AjIG5vdCBqdXN0IGRpc2NvdmVyZWQgYXQgdGhlIGVuZC4gQ29uZmlnIGZpZWxkcyBhcmUgd3JpdHRlbiBieSB0aGUKICAgICAgICAjIG5vdGVi'
        'b29rJ3Mgd3JpdGVfY29uZmlnKCk7IGlmIHRoZXkncmUgYWJzZW50IChlLmcuIGEgcGxhaW4gYmFzZWxpbmUvCiAgICAgICAgIyBjb250cm9sL3'
        'Byb3Bvc2VkIGV2YWwgcnVuLCBub3QgYSB0cmFpbmluZyBydW4pIHRoaXMgaXMgYSBuby1vcC4KICAgICAgICAjIFdyYXBwZWQgaW4gdHJ5L2V4'
        'Y2VwdCBzbyBhIGZhaWx1cmUgaGVyZSAoT09NLCBiYWQgcGF0aCwgZXRjLikgY2FuCiAgICAgICAgIyBuZXZlciB0YWtlIGRvd24gYW4gaW4tcH'
        'JvZ3Jlc3MgdHJhaW5pbmcgcnVuLgogICAgICAgIHRyeToKICAgICAgICAgICAgZnJvbSBjb25maWcgaW1wb3J0IChSRUFMX0VWQUxfUEVSSU9E'
        'LCBWT0NGT0dfVEVTVF9JTUFHRVNfRElSLCBWT0NGT0dfVEVTVF9BTk5fRElSLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBSVF'
        'RTX0lNQUdFU19ESVIsIFJUVFNfQU5OX0RJUiwgUlRUU19JRFNfRklMRSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY2xhc3Nl'
        'c19wYXRoIGFzIF9ldmFsX2NsYXNzZXNfcGF0aCwgYW5jaG9yc19wYXRoIGFzIF9ldmFsX2FuY2hvcnNfcGF0aCwKICAgICAgICAgICAgICAgIC'
        'AgICAgICAgICAgICAgICAgQkFTRUxJTkVfVk9DRk9HX01BUCwgQkFTRUxJTkVfUlRUU19NQVApCiAgICAgICAgICAgIF9yZWFsX2V2YWxfZW5h'
        'YmxlZCA9IFRydWUKICAgICAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgICAgIF9yZWFsX2V2YWxfZW5hYmxlZCA9IEZhbHNlCgogIC'
        'AgICAgIGlmIF9yZWFsX2V2YWxfZW5hYmxlZCBhbmQgKChlcG9jaCArIDEpICUgUkVBTF9FVkFMX1BFUklPRCA9PSAwIG9yIGVwb2NoICsgMSA9'
        'PSBFcG9jaCk6CiAgICAgICAgICAgIGltcG9ydCBzdWJwcm9jZXNzLCBqc29uIGFzIF9qc29uCiAgICAgICAgICAgIHRyeToKICAgICAgICAgIC'
        'AgICAgIHRtcF9ja3B0ID0gb3MucGF0aC5qb2luKHNhdmVfZGlyLCAndG1wX3JlYWxfZXZhbC5wdGgnKQogICAgICAgICAgICAgICAgdG9yY2gu'
        'c2F2ZShzYXZlX3N0YXRlX2RpY3QsIHRtcF9ja3B0KQogICAgICAgICAgICAgICAgcmVzdWx0cyA9IHt9CiAgICAgICAgICAgICAgICBmb3IgZG'
        'F0YW5hbWUsIGltYWdlc19kaXIsIGFubl9kaXIsIGV4dCwgaWRzX2ZpbGUgaW4gWwogICAgICAgICAgICAgICAgICAgICgnVk9DLUZPRy10ZXN0'
        'JywgVk9DRk9HX1RFU1RfSU1BR0VTX0RJUiwgVk9DRk9HX1RFU1RfQU5OX0RJUiwgJy5qcGcnLCBOb25lKSwKICAgICAgICAgICAgICAgICAgIC'
        'AoJ1JUVFMnLCBSVFRTX0lNQUdFU19ESVIsIFJUVFNfQU5OX0RJUiwgJy5wbmcnLCBSVFRTX0lEU19GSUxFKSwKICAgICAgICAgICAgICAgIF06'
        'CiAgICAgICAgICAgICAgICAgICAgb3V0X2pzb24gPSBvcy5wYXRoLmpvaW4oc2F2ZV9kaXIsIGYncmVhbF9ldmFsX2Vwe2Vwb2NoICsgMX1fe2'
        'RhdGFuYW1lfS5qc29uJykKICAgICAgICAgICAgICAgICAgICBjbWQgPSBbJ3B5dGhvbicsICdldmFsX29uZS5weScsICctLWRhdGFuYW1lJywg'
        'ZGF0YW5hbWUsICctLW1vZGVsX3BhdGgnLCB0bXBfY2twdCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgJy0taW1hZ2VzX2RpcicsIGltYW'
        'dlc19kaXIsICctLWFubl9kaXInLCBhbm5fZGlyLCAnLS1pbWFnZV9leHQnLCBleHQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICctLWNs'
        'YXNzZXNfcGF0aCcsIF9ldmFsX2NsYXNzZXNfcGF0aCwgJy0tYW5jaG9yc19wYXRoJywgX2V2YWxfYW5jaG9yc19wYXRoLAogICAgICAgICAgIC'
        'AgICAgICAgICAgICAgICAnLS1vdXRfanNvbicsIG91dF9qc29uXQogICAgICAgICAgICAgICAgICAgIGlmIGlkc19maWxlOgogICAgICAgICAg'
        'ICAgICAgICAgICAgICBjbWQgKz0gWyctLWlkc19maWxlJywgaWRzX2ZpbGVdCiAgICAgICAgICAgICAgICAgICAgcHJpbnQoZidbcmVhbC1ldm'
        'FsXSBlcG9jaCB7ZXBvY2ggKyAxfTogcnVubmluZyB7ZGF0YW5hbWV9IG9uIHRoZSByZWFsIHRlc3Qgc2V0Li4uJykKICAgICAgICAgICAgICAg'
        'ICAgICByZXQgPSBzdWJwcm9jZXNzLnJ1bihjbWQsIGNhcHR1cmVfb3V0cHV0PVRydWUsIHRleHQ9VHJ1ZSkKICAgICAgICAgICAgICAgICAgIC'
        'BpZiByZXQucmV0dXJuY29kZSAhPSAwOgogICAgICAgICAgICAgICAgICAgICAgICBwcmludChmJ1tyZWFsLWV2YWxdIHtkYXRhbmFtZX0gRkFJ'
        'TEVEIGF0IGVwb2NoIHtlcG9jaCArIDF9IChjb250aW51aW5nIHRyYWluaW5nKTonKQogICAgICAgICAgICAgICAgICAgICAgICBwcmludChyZX'
        'Quc3Rkb3V0Wy0xNTAwOl0pCiAgICAgICAgICAgICAgICAgICAgICAgIHByaW50KHJldC5zdGRlcnJbLTE1MDA6XSkKICAgICAgICAgICAgICAg'
        'ICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgICAgICB3aXRoIG9wZW4ob3V0X2pzb24pIGFzIGY6CiAgICAgICAgICAgICAgICAgIC'
        'AgICAgIHJlc3VsdHNbZGF0YW5hbWVdID0gX2pzb24ubG9hZChmKQoKICAgICAgICAgICAgICAgIGlmICdWT0MtRk9HLXRlc3QnIGluIHJlc3Vs'
        'dHM6CiAgICAgICAgICAgICAgICAgICAgbSA9IHJlc3VsdHNbJ1ZPQy1GT0ctdGVzdCddWydtQVAnXQogICAgICAgICAgICAgICAgICAgIGlmIE'
        'JBU0VMSU5FX1ZPQ0ZPR19NQVAgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgICAgIHByaW50KCdbcmVhbC1ldmFsXSBlcG9jaCAl'
        'ZCB8IFZPQy1GT0ctdGVzdCBtQVA9JS4yZiUlIHwgYmFzZWxpbmU9JS4yZiUlIHwgZGVsdGE9JSsuMmYlJScKICAgICAgICAgICAgICAgICAgIC'
        'AgICAgICAgICAgJSAoZXBvY2ggKyAxLCBtLCBCQVNFTElORV9WT0NGT0dfTUFQLCBtIC0gQkFTRUxJTkVfVk9DRk9HX01BUCkpCiAgICAgICAg'
        'ICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICAgICAgcHJpbnQoJ1tyZWFsLWV2YWxdIGVwb2NoICVkIHwgVk9DLUZPRy10ZX'
        'N0IG1BUD0lLjJmJSUgfCAobm8gYmFzZWxpbmUgY29uZmlndXJlZCknCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICUgKGVwb2NoICsg'
        'MSwgbSkpCiAgICAgICAgICAgICAgICBpZiAnUlRUUycgaW4gcmVzdWx0czoKICAgICAgICAgICAgICAgICAgICBtID0gcmVzdWx0c1snUlRUUy'
        'ddWydtQVAnXQogICAgICAgICAgICAgICAgICAgIGlmIEJBU0VMSU5FX1JUVFNfTUFQIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAg'
        'ICAgICBwcmludCgnW3JlYWwtZXZhbF0gZXBvY2ggJWQgfCBSVFRTIG1BUD0lLjJmJSUgfCBiYXNlbGluZT0lLjJmJSUgfCBkZWx0YT0lKy4yZi'
        'UlJwogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAlIChlcG9jaCArIDEsIG0sIEJBU0VMSU5FX1JUVFNfTUFQLCBtIC0gQkFTRUxJTkVf'
        'UlRUU19NQVApKQogICAgICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgICAgIHByaW50KCdbcmVhbC1ldmFsXSBlcG'
        '9jaCAlZCB8IFJUVFMgbUFQPSUuMmYlJSB8IChubyBiYXNlbGluZSBjb25maWd1cmVkKScKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAg'
        'JSAoZXBvY2ggKyAxLCBtKSkKCiAgICAgICAgICAgICAgICBoaXN0b3J5X3BhdGggPSBvcy5wYXRoLmpvaW4oc2F2ZV9kaXIsICdyZWFsX2V2YW'
        'xfaGlzdG9yeS5qc29uJykKICAgICAgICAgICAgICAgIGhpc3RvcnkgPSBbXQogICAgICAgICAgICAgICAgaWYgb3MucGF0aC5leGlzdHMoaGlz'
        'dG9yeV9wYXRoKToKICAgICAgICAgICAgICAgICAgICB3aXRoIG9wZW4oaGlzdG9yeV9wYXRoKSBhcyBmOgogICAgICAgICAgICAgICAgICAgIC'
        'AgICBoaXN0b3J5ID0gX2pzb24ubG9hZChmKQogICAgICAgICAgICAgICAgaGlzdG9yeS5hcHBlbmQoeydlcG9jaCc6IGVwb2NoICsgMSwgKip7'
        'azogdi5nZXQoJ21BUCcpIGZvciBrLCB2IGluIHJlc3VsdHMuaXRlbXMoKX19KQogICAgICAgICAgICAgICAgd2l0aCBvcGVuKGhpc3RvcnlfcG'
        'F0aCwgJ3cnKSBhcyBmOgogICAgICAgICAgICAgICAgICAgIF9qc29uLmR1bXAoaGlzdG9yeSwgZiwgaW5kZW50PTIpCiAgICAgICAgICAgICAg'
        'ICBpZiBvcy5wYXRoLmV4aXN0cyh0bXBfY2twdCk6CiAgICAgICAgICAgICAgICAgICAgb3MucmVtb3ZlKHRtcF9ja3B0KQogICAgICAgICAgIC'
        'BleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgICAgICBwcmludChmJ1tyZWFsLWV2YWxdIFdBUk5JTkc6IHBlcmlvZGljIHJlYWwt'
        'dGVzdCBldmFsIGZhaWxlZCBhdCBlcG9jaCB7ZXBvY2ggKyAxfToge2Uhcn0gJwogICAgICAgICAgICAgICAgICAgICAgZictLSB0cmFpbmluZy'
        'Bjb250aW51ZXMuJykK'
    ),
    'eval_one.py': (
        'IiIiClN0YW5kYWxvbmUsIHN1YnByb2Nlc3MtaXNvbGF0ZWQgZXZhbHVhdGlvbiBzY3JpcHQuCgpXaHkgdGhpcyBleGlzdHMgYXMgYSBzZXBhcm'
        'F0ZSBzY3JpcHQgaW5zdGVhZCBvZiBhIGZ1bmN0aW9uIGltcG9ydGVkIGludG8gdGhlCm5vdGVib29rJ3MgbG9uZy1saXZlZCBrZXJuZWw6IHRo'
        'aXMgcHJvamVjdCdzIGFyY2hpdGVjdHVyZSAobmV0cy9tb2RlbC5weSkKcmVhZHMgY29uZmlnLlVTRV9EUk0gYXQgaW1wb3J0IHRpbWUgdG8gZG'
        'VjaWRlIHdoZXRoZXIgdGhlIFAzIGJyYW5jaCBpbmNsdWRlcwp0aGUgRGV0YWlsIFJlY292ZXJ5IE1vZHVsZS4gV2l0aGluIG9uZSBub3RlYm9v'
        'ayBzZXNzaW9uIHdlIGV2YWx1YXRlCmNoZWNrcG9pbnRzIHRyYWluZWQgdW5kZXIgZGlmZmVyZW50IFVTRV9EUk0gc2V0dGluZ3MgKHRoZSB1bm'
        'NoYW5nZWQtYmFzZWxpbmUKY29udHJvbCB2cy4gdGhlIERSTStXaXNlLUlvVStEV0EgcHJvcG9zZWQgc3lzdGVtKSAtLSBpZiBib3RoIHdlcmUg'
        'ZXZhbHVhdGVkCnZpYSBgaW1wb3J0IG5ldHMubW9kZWxgIGluIHRoZSBzYW1lIFB5dGhvbiBwcm9jZXNzLCBvbmx5IHRoZSBGSVJTVCBpbXBvcn'
        'QKd291bGQgYWN0dWFsbHkgdGFrZSBlZmZlY3QgKFB5dGhvbiBjYWNoZXMgbW9kdWxlcyksIHNpbGVudGx5IGV2YWx1YXRpbmcgdGhlCndyb25n'
        'IGFyY2hpdGVjdHVyZSBmb3IgdGhlIHNlY29uZCBjaGVja3BvaW50LiBSdW5uaW5nIGVhY2ggZXZhbHVhdGlvbiBhcyBhCmZyZXNoIGBweXRob2'
        '4gZXZhbF9vbmUucHkgLi4uYCBzdWJwcm9jZXNzIHNpZGVzdGVwcyB0aGF0IGVudGlyZWx5OiBldmVyeQppbnZvY2F0aW9uIHJlLXJlYWRzIGNv'
        'bmZpZy5weSBmcm9tIGRpc2ssIHNvIGl0IGFsd2F5cyBidWlsZHMgdGhlIGV4YWN0CmFyY2hpdGVjdHVyZSB0aGUgY2hlY2twb2ludCBiZWluZy'
        'BldmFsdWF0ZWQgd2FzIHRyYWluZWQgd2l0aC4KCkJlY2F1c2Ugb2YgdGhpcywgQUxXQVlTIChyZSl3cml0ZSBjb25maWcucHkgd2l0aCB0aGUg'
        'Y29ycmVjdCBVU0VfRFJNIHZhbHVlCmZvciB0aGUgY2hlY2twb2ludCB5b3UgYXJlIGFib3V0IHRvIGV2YWx1YXRlIGJlZm9yZSBjYWxsaW5nIH'
        'RoaXMgc2NyaXB0LgoKVXNhZ2U6CiAgICBweXRob24gZXZhbF9vbmUucHkgLS1kYXRhbmFtZSBOQU1FIC0tbW9kZWxfcGF0aCBQQVRIIFwKICAg'
        'ICAgICAtLWltYWdlc19kaXIgRElSIC0tYW5uX2RpciBESVIgLS1pbWFnZV9leHQgLmpwZyBcCiAgICAgICAgLS1vdXRfanNvbiByZXN1bHQuan'
        'NvbiBcCiAgICAgICAgWy0taWRzX2ZpbGUgcGF0aF93aXRoX29uZV9pZF9wZXJfbGluZV0gXAogICAgICAgIFstLWNsYXNzZXNfcGF0aCBtb2Rl'
        'bF9kYXRhL3J0dHNfY2xhc3Nlcy50eHRdIFwKICAgICAgICBbLS1hbmNob3JzX3BhdGggbW9kZWxfZGF0YS95b2xvX2FuY2hvcnMudHh0XQoKSW'
        'YgLS1pZHNfZmlsZSBpcyBvbWl0dGVkLCBldmVyeSBmaWxlIGluIC0taW1hZ2VzX2RpciBtYXRjaGluZyAtLWltYWdlX2V4dCBpcwp1c2VkIGFz'
        'IHRoZSBpZCBsaXN0IChpZCA9IGZpbGVuYW1lIHdpdGhvdXQgZXh0ZW5zaW9uKS4KIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbX'
        'BvcnQgb3MKaW1wb3J0IHJlCmltcG9ydCB4bWwuZXRyZWUuRWxlbWVudFRyZWUgYXMgRVQKCmltcG9ydCB0b3JjaApmcm9tIFBJTCBpbXBvcnQg'
        'SW1hZ2UsIFVuaWRlbnRpZmllZEltYWdlRXJyb3IKZnJvbSB0cWRtIGltcG9ydCB0cWRtCgpmcm9tIHV0aWxzLnV0aWxzIGltcG9ydCBnZXRfY2'
        'xhc3Nlcwpmcm9tIHV0aWxzLnV0aWxzX21hcCBpbXBvcnQgZ2V0X21hcApmcm9tIHlvbG8gaW1wb3J0IFlPTE8KCgpkZWYgbGlzdF9pZHNfZnJv'
        'bV9kaXIoaW1hZ2VzX2RpciwgZXh0KToKICAgIHJldHVybiBzb3J0ZWQob3MucGF0aC5zcGxpdGV4dChmKVswXSBmb3IgZiBpbiBvcy5saXN0ZG'
        'lyKGltYWdlc19kaXIpIGlmIGYubG93ZXIoKS5lbmRzd2l0aChleHQpKQoKCmRlZiBwYXJzZV9wZXJfY2xhc3NfYXAocmVzdWx0c190eHRfcGF0'
        'aCk6CiAgICAiIiJ1dGlsc19tYXAuZ2V0X21hcCB3cml0ZXMgbGluZXMgbGlrZSAnNjYuMzUlID0gYmljeWNsZSBBUCAnIGludG8KICAgIHJlc3'
        'VsdHMudHh0IGluc2lkZSBtYXBfb3V0X3BhdGggLS0gcHVsbCB0aG9zZSBiYWNrIG91dCBpbnN0ZWFkIG9mCiAgICBtb2RpZnlpbmcgdXRpbHNf'
        'bWFwLnB5J3MgYWxyZWFkeS13b3JraW5nIChhbmQgZW50YW5nbGVkKSBpbnRlcm5hbHMuIiIiCiAgICBwZXJfY2xhc3MgPSB7fQogICAgaWYgbm'
        '90IG9zLnBhdGguZXhpc3RzKHJlc3VsdHNfdHh0X3BhdGgpOgogICAgICAgIHJldHVybiBwZXJfY2xhc3MKICAgIHBhdHRlcm4gPSByZS5jb21w'
        'aWxlKHInXihbXGQuXSspJSA9IChcUyspIEFQXGInKQogICAgd2l0aCBvcGVuKHJlc3VsdHNfdHh0X3BhdGgpIGFzIGY6CiAgICAgICAgZm9yIG'
        'xpbmUgaW4gZjoKICAgICAgICAgICAgbSA9IHBhdHRlcm4ubWF0Y2gobGluZS5zdHJpcCgpKQogICAgICAgICAgICBpZiBtOgogICAgICAgICAg'
        'ICAgICAgcGVyX2NsYXNzW20uZ3JvdXAoMildID0gZmxvYXQobS5ncm91cCgxKSkKICAgIHJldHVybiBwZXJfY2xhc3MKCgpkZWYgZXZhbHVhdG'
        'UoZGF0YW5hbWUsIGltYWdlX2lkcywgaW1hZ2VzX2RpciwgYW5ub3RhdGlvbnNfZGlyLCBtb2RlbF9wYXRoLCBpbWFnZV9leHQsCiAgICAgICAg'
        'ICAgICBjbGFzc2VzX3BhdGgsIGFuY2hvcnNfcGF0aCwgbWluX292ZXJsYXA9MC41LCBjb25maWRlbmNlPTAuMDAxLAogICAgICAgICAgICAgbm'
        '1zX2lvdT0wLjUsIHNjb3JlX3RocmVzaG9sZD0wLjUsIG1hcF9vdXRfcGF0aD1Ob25lKToKICAgIG1hcF9vdXRfcGF0aCA9IG1hcF9vdXRfcGF0'
        'aCBvciBmJ21hcF9vdXQte2RhdGFuYW1lfScKICAgIGZvciBzdWIgaW4gWydncm91bmQtdHJ1dGgnLCAnZGV0ZWN0aW9uLXJlc3VsdHMnLCAnaW'
        '1hZ2VzLW9wdGlvbmFsJ106CiAgICAgICAgb3MubWFrZWRpcnMob3MucGF0aC5qb2luKG1hcF9vdXRfcGF0aCwgc3ViKSwgZXhpc3Rfb2s9VHJ1'
        'ZSkKCiAgICBjbGFzc19uYW1lcywgXyA9IGdldF9jbGFzc2VzKGNsYXNzZXNfcGF0aCkKICAgIHByaW50KGYiW3tkYXRhbmFtZX1dIHtsZW4oaW'
        '1hZ2VfaWRzKX0gaW1hZ2VzIHwgbG9hZGluZyBtb2RlbCBmcm9tIHttb2RlbF9wYXRofSIpCiAgICB5b2xvID0gWU9MTyhtb2RlbF9wYXRoPW1v'
        'ZGVsX3BhdGgsIGNsYXNzZXNfcGF0aD1jbGFzc2VzX3BhdGgsIGFuY2hvcnNfcGF0aD1hbmNob3JzX3BhdGgsCiAgICAgICAgICAgICAgICBjb2'
        '5maWRlbmNlPWNvbmZpZGVuY2UsIG5tc19pb3U9bm1zX2lvdSwgY3VkYT10b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpKQoKICAgIHNraXBwZWQg'
        'PSBbXQogICAgZm9yIGltYWdlX2lkIGluIHRxZG0oaW1hZ2VfaWRzLCBkZXNjPWYnW3tkYXRhbmFtZX1dIHByZWRpY3RpbmcnKToKICAgICAgIC'
        'BpbWFnZV9wYXRoID0gb3MucGF0aC5qb2luKGltYWdlc19kaXIsIGltYWdlX2lkICsgaW1hZ2VfZXh0KQogICAgICAgIHRyeToKICAgICAgICAg'
        'ICAgaW1hZ2UgPSBJbWFnZS5vcGVuKGltYWdlX3BhdGgpCiAgICAgICAgICAgIGltYWdlLmxvYWQoKQogICAgICAgIGV4Y2VwdCAoVW5pZGVudG'
        'lmaWVkSW1hZ2VFcnJvciwgT1NFcnJvcikgYXMgZToKICAgICAgICAgICAgc2tpcHBlZC5hcHBlbmQoKGltYWdlX2lkLCBzdHIoZSkpKQogICAg'
        'ICAgICAgICBvcGVuKG9zLnBhdGguam9pbihtYXBfb3V0X3BhdGgsIGYiZGV0ZWN0aW9uLXJlc3VsdHMve2ltYWdlX2lkfS50eHQiKSwgInciKS'
        '5jbG9zZSgpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgeW9sby5nZXRfbWFwX3R4dChpbWFnZV9pZCwgaW1hZ2UsIGNsYXNzX25hbWVz'
        'LCBtYXBfb3V0X3BhdGgpCgogICAgaWYgc2tpcHBlZDoKICAgICAgICBwcmludChmIlt7ZGF0YW5hbWV9XSBXQVJOSU5HOiBza2lwcGVkIHtsZW'
        '4oc2tpcHBlZCl9L3tsZW4oaW1hZ2VfaWRzKX0gdW5yZWFkYWJsZSBpbWFnZShzKSAiCiAgICAgICAgICAgICAgZiIodHJlYXRlZCBhcyB6ZXJv'
        'IGRldGVjdGlvbnMpIikKICAgICAgICBmb3IgaW1hZ2VfaWQsIGVyciBpbiBza2lwcGVkWzoyMF06CiAgICAgICAgICAgIHByaW50KGYiICAgIH'
        'tpbWFnZV9pZH17aW1hZ2VfZXh0fToge2Vycn0iKQoKICAgIGJhZF94bWwgPSBbXQogICAgZm9yIGltYWdlX2lkIGluIHRxZG0oaW1hZ2VfaWRz'
        'LCBkZXNjPWYnW3tkYXRhbmFtZX1dIGdyb3VuZCB0cnV0aCcpOgogICAgICAgIHdpdGggb3Blbihvcy5wYXRoLmpvaW4obWFwX291dF9wYXRoLC'
        'BmImdyb3VuZC10cnV0aC97aW1hZ2VfaWR9LnR4dCIpLCAidyIpIGFzIG5ld19mOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBy'
        'b290ID0gRVQucGFyc2Uob3MucGF0aC5qb2luKGFubm90YXRpb25zX2RpciwgZiJ7aW1hZ2VfaWR9LnhtbCIpKS5nZXRyb290KCkKICAgICAgIC'
        'AgICAgZXhjZXB0IEVULlBhcnNlRXJyb3IgYXMgZToKICAgICAgICAgICAgICAgIGJhZF94bWwuYXBwZW5kKChpbWFnZV9pZCwgc3RyKGUpKSkK'
        'ICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGZvciBvYmogaW4gcm9vdC5maW5kYWxsKCdvYmplY3QnKToKICAgICAgICAgIC'
        'AgICAgIGRpZmZpY3VsdF9mbGFnID0gb2JqLmZpbmQoJ2RpZmZpY3VsdCcpIGlzIG5vdCBOb25lIGFuZCBpbnQob2JqLmZpbmQoJ2RpZmZpY3Vs'
        'dCcpLnRleHQpID09IDEKICAgICAgICAgICAgICAgIG9ial9uYW1lID0gb2JqLmZpbmQoJ25hbWUnKS50ZXh0CiAgICAgICAgICAgICAgICBpZi'
        'BvYmpfbmFtZSBub3QgaW4gY2xhc3NfbmFtZXM6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIGJuZCA9IG9i'
        'ai5maW5kKCdibmRib3gnKQogICAgICAgICAgICAgICAgbGVmdCwgdG9wID0gYm5kLmZpbmQoJ3htaW4nKS50ZXh0LCBibmQuZmluZCgneW1pbi'
        'cpLnRleHQKICAgICAgICAgICAgICAgIHJpZ2h0LCBib3R0b20gPSBibmQuZmluZCgneG1heCcpLnRleHQsIGJuZC5maW5kKCd5bWF4JykudGV4'
        'dAogICAgICAgICAgICAgICAgc3VmZml4ID0gIiBkaWZmaWN1bHQiIGlmIGRpZmZpY3VsdF9mbGFnIGVsc2UgIiIKICAgICAgICAgICAgICAgIG'
        '5ld19mLndyaXRlKGYie29ial9uYW1lfSB7bGVmdH0ge3RvcH0ge3JpZ2h0fSB7Ym90dG9tfXtzdWZmaXh9XG4iKQoKICAgIGlmIGJhZF94bWw6'
        'CiAgICAgICAgcHJpbnQoZiJbe2RhdGFuYW1lfV0gV0FSTklORzoge2xlbihiYWRfeG1sKX0gdW5wYXJzZWFibGUgYW5ub3RhdGlvbihzKSAodH'
        'JlYXRlZCBhcyAwIG9iamVjdHMpIikKCiAgICBwcmludChmIlt7ZGF0YW5hbWV9XSBjb21wdXRpbmcgbUFQIikKICAgICMgTk9URTogZ2V0X21h'
        'cCByZXR1cm5zIG1BUCBhcyBhIDAtMSBmcmFjdGlvbiAoZS5nLiAwLjc4MzkpLCBub3QgYQogICAgIyBwZXJjZW50YWdlIC0tIGRlc3BpdGUgdG'
        'hlIGV4aXN0aW5nIG5vdGVib29rcycgZXZhbHVhdGUoKSBwcmludGluZyBpdAogICAgIyB3aXRoIGEgdHJhaWxpbmcgJyUnIHVuc2NhbGVkIChh'
        'IHByZS1leGlzdGluZyBmb3JtYXR0aW5nIGluY29uc2lzdGVuY3kKICAgICMgaW4gdGhpcyByZXBvLCBub3QgaW50cm9kdWNlZCBoZXJlKS4gV2'
        'UgbXVsdGlwbHkgYnkgMTAwIGV4cGxpY2l0bHkgYmVsb3cKICAgICMgc28gcmVzdWx0cyBpbiBvdXRfanNvbiBhcmUgdW5hbWJpZ3VvdXNseSBw'
        'ZXJjZW50YWdlcy4KICAgIG1BUF9mcmFjdGlvbiA9IGdldF9tYXAobWluX292ZXJsYXAsIFRydWUsIHNjb3JlX3RocmVob2xkPXNjb3JlX3Rocm'
        'VzaG9sZCwgcGF0aD1tYXBfb3V0X3BhdGgpCiAgICBwZXJfY2xhc3MgPSBwYXJzZV9wZXJfY2xhc3NfYXAob3MucGF0aC5qb2luKG1hcF9vdXRf'
        'cGF0aCwgJ3Jlc3VsdHMudHh0JykpCiAgICByZXR1cm4gbUFQX2ZyYWN0aW9uICogMTAwLjAsIHBlcl9jbGFzcwoKCmRlZiBtYWluKCk6CiAgIC'
        'BwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgcC5hZGRfYXJndW1lbnQoJy0tZGF0YW5hbWUnLCByZXF1aXJlZD1UcnVlKQogICAg'
        'cC5hZGRfYXJndW1lbnQoJy0tbW9kZWxfcGF0aCcsIHJlcXVpcmVkPVRydWUpCiAgICBwLmFkZF9hcmd1bWVudCgnLS1pbWFnZXNfZGlyJywgcm'
        'VxdWlyZWQ9VHJ1ZSkKICAgIHAuYWRkX2FyZ3VtZW50KCctLWFubl9kaXInLCByZXF1aXJlZD1UcnVlKQogICAgcC5hZGRfYXJndW1lbnQoJy0t'
        'aW1hZ2VfZXh0JywgZGVmYXVsdD0nLmpwZycpCiAgICBwLmFkZF9hcmd1bWVudCgnLS1pZHNfZmlsZScsIGRlZmF1bHQ9Tm9uZSwKICAgICAgIC'
        'AgICAgICAgICAgICBoZWxwPSdvcHRpb25hbCBmaWxlIHdpdGggb25lIGltYWdlIGlkIHBlciBsaW5lOyBkZWZhdWx0cyB0byBsaXN0aW5nIGlt'
        'YWdlc19kaXInKQogICAgcC5hZGRfYXJndW1lbnQoJy0tY2xhc3Nlc19wYXRoJywgZGVmYXVsdD0nbW9kZWxfZGF0YS9ydHRzX2NsYXNzZXMudH'
        'h0JykKICAgIHAuYWRkX2FyZ3VtZW50KCctLWFuY2hvcnNfcGF0aCcsIGRlZmF1bHQ9J21vZGVsX2RhdGEveW9sb19hbmNob3JzLnR4dCcpCiAg'
        'ICBwLmFkZF9hcmd1bWVudCgnLS1taW5fb3ZlcmxhcCcsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC41KQogICAgcC5hZGRfYXJndW1lbnQoJy0tY2'
        '9uZmlkZW5jZScsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4wMDEpCiAgICBwLmFkZF9hcmd1bWVudCgnLS1ubXNfaW91JywgdHlwZT1mbG9hdCwg'
        'ZGVmYXVsdD0wLjUpCiAgICBwLmFkZF9hcmd1bWVudCgnLS1zY29yZV90aHJlc2hvbGQnLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuNSkKICAgIH'
        'AuYWRkX2FyZ3VtZW50KCctLW91dF9qc29uJywgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFyZ3MgPSBwLnBhcnNlX2FyZ3MoKQoKICAgIGlmIGFyZ3Mu'
        'aWRzX2ZpbGU6CiAgICAgICAgIyBNYXRjaGVzIHJkZm5ldF9iYXNlbGluZV9ldmFsLmlweW5iJ3MgcmVhZF90ZXN0X2lkcygpIGNvbnZlbnRpb2'
        '4gZXhhY3RseQogICAgICAgICMgKHdoaXRlc3BhY2Utc3BsaXQsIG5vdCBsaW5lLXNwbGl0KSBmb3IgY29uc2lzdGVuY3kgd2l0aCB0aGUgYWxy'
        'ZWFkeS0KICAgICAgICAjIHZhbGlkYXRlZCBiYXNlbGluZSBldmFsdWF0aW9uLgogICAgICAgIGltYWdlX2lkcyA9IG9wZW4oYXJncy5pZHNfZm'
        'lsZSkucmVhZCgpLnN0cmlwKCkuc3BsaXQoKQogICAgZWxzZToKICAgICAgICBpbWFnZV9pZHMgPSBsaXN0X2lkc19mcm9tX2RpcihhcmdzLmlt'
        'YWdlc19kaXIsIGFyZ3MuaW1hZ2VfZXh0KQoKICAgIG1BUCwgcGVyX2NsYXNzID0gZXZhbHVhdGUoCiAgICAgICAgZGF0YW5hbWU9YXJncy5kYX'
        'RhbmFtZSwgaW1hZ2VfaWRzPWltYWdlX2lkcywgaW1hZ2VzX2Rpcj1hcmdzLmltYWdlc19kaXIsCiAgICAgICAgYW5ub3RhdGlvbnNfZGlyPWFy'
        'Z3MuYW5uX2RpciwgbW9kZWxfcGF0aD1hcmdzLm1vZGVsX3BhdGgsIGltYWdlX2V4dD1hcmdzLmltYWdlX2V4dCwKICAgICAgICBjbGFzc2VzX3'
        'BhdGg9YXJncy5jbGFzc2VzX3BhdGgsIGFuY2hvcnNfcGF0aD1hcmdzLmFuY2hvcnNfcGF0aCwKICAgICAgICBtaW5fb3ZlcmxhcD1hcmdzLm1p'
        'bl9vdmVybGFwLCBjb25maWRlbmNlPWFyZ3MuY29uZmlkZW5jZSwKICAgICAgICBubXNfaW91PWFyZ3Mubm1zX2lvdSwgc2NvcmVfdGhyZXNob2'
        'xkPWFyZ3Muc2NvcmVfdGhyZXNob2xkLAogICAgICAgIG1hcF9vdXRfcGF0aD1mJy9rYWdnbGUvd29ya2luZy9tYXBfb3V0LXthcmdzLmRhdGFu'
        'YW1lfScsCiAgICApCgogICAgcmVzdWx0ID0geydkYXRhbmFtZSc6IGFyZ3MuZGF0YW5hbWUsICdtb2RlbF9wYXRoJzogYXJncy5tb2RlbF9wYX'
        'RoLAogICAgICAgICAgICAgICdudW1faW1hZ2VzJzogbGVuKGltYWdlX2lkcyksICdtQVAnOiBtQVAsICdwZXJfY2xhc3NfQVAnOiBwZXJfY2xh'
        'c3N9CiAgICB3aXRoIG9wZW4oYXJncy5vdXRfanNvbiwgJ3cnKSBhcyBmOgogICAgICAgIGpzb24uZHVtcChyZXN1bHQsIGYsIGluZGVudD0yKQ'
        'ogICAgcHJpbnQoZiJcbj09PSBbe2FyZ3MuZGF0YW5hbWV9XSBtQVBAe2FyZ3MubWluX292ZXJsYXB9OiB7bUFQOi4yZn0lID09PSIpCiAgICBm'
        'b3IgY2xzLCBhcCBpbiBzb3J0ZWQocGVyX2NsYXNzLml0ZW1zKCkpOgogICAgICAgIHByaW50KGYiICAgIHtjbHM6MTJzfToge2FwOi4yZn0lIi'
        'kKICAgIHByaW50KGYiUmVzdWx0IHdyaXR0ZW4gdG8ge2FyZ3Mub3V0X2pzb259IikKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAg'
        'bWFpbigpCg=='
    ),
    'train.py': (
        'aW1wb3J0IGRhdGV0aW1lCmltcG9ydCBvcwpmcm9tIGZ1bmN0b29scyBpbXBvcnQgcGFydGlhbAojIG9zLmVudmlyb25bIkNVREFfVklTSUJMRV'
        '9ERVZJQ0VTIl0gPSAiMCIKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2guYmFja2VuZHMuY3Vkbm4gYXMgY3Vk'
        'bm4KaW1wb3J0IHRvcmNoLmRpc3RyaWJ1dGVkIGFzIGRpc3QKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmltcG9ydCB0b3JjaC5vcHRpbSBhcyBvcH'
        'RpbQpmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFMb2FkZXIKZnJvbSBuZXRzLm1vZGVsIGltcG9ydCBZb2xvQm9keQpmcm9tIG5l'
        'dHMueW9sb190cmFpbmluZyBpbXBvcnQgKE1vZGVsRU1BLCBZT0xPTG9zcywgZ2V0X2xyX3NjaGVkdWxlciwKICAgICAgICAgICAgICAgICAgIC'
        'AgICAgICAgICAgICBzZXRfb3B0aW1pemVyX2xyLCB3ZWlnaHRzX2luaXQpCmZyb20gdXRpbHMuY2FsbGJhY2tzIGltcG9ydCBFdmFsQ2FsbGJh'
        'Y2ssIExvc3NIaXN0b3J5CmZyb20gdXRpbHMuZGF0YWxvYWRlciBpbXBvcnQgWW9sb0RhdGFzZXQsIHlvbG9fZGF0YXNldF9jb2xsYXRlCmZyb2'
        '0gdXRpbHMudXRpbHMgaW1wb3J0IChnZXRfYW5jaG9ycywgZ2V0X2NsYXNzZXMsCiAgICAgICAgICAgICAgICAgICAgICAgICBzZWVkX2V2ZXJ5'
        'dGhpbmcsIHNob3dfY29uZmlnLCB3b3JrZXJfaW5pdF9mbikKZnJvbSB1dGlscy51dGlsc19maXQgaW1wb3J0IGZpdF9vbmVfZXBvY2gKZnJvbS'
        'Bjb25maWcgaW1wb3J0ICoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICAKICAgIHNlZWRfZXZlcnl0aGluZyhzZWVkKQogICAgbmdw'
        'dXNfcGVyX25vZGUgID0gdG9yY2guY3VkYS5kZXZpY2VfY291bnQoKQogICAgaWYgZGlzdHJpYnV0ZWQ6CiAgICAgICAgZGlzdC5pbml0X3Byb2'
        'Nlc3NfZ3JvdXAoYmFja2VuZD0ibmNjbCIpCiAgICAgICAgbG9jYWxfcmFuayAgPSBpbnQob3MuZW52aXJvblsiTE9DQUxfUkFOSyJdKQogICAg'
        'ICAgIHJhbmsgICAgICAgID0gaW50KG9zLmVudmlyb25bIlJBTksiXSkKICAgICAgICBkZXZpY2UgICAgICA9IHRvcmNoLmRldmljZSgiY3VkYS'
        'IsIGxvY2FsX3JhbmspCiAgICAgICAgaWYgbG9jYWxfcmFuayA9PSAwOgogICAgICAgICAgICBwcmludChmIlt7b3MuZ2V0cGlkKCl9XSAocmFu'
        'ayA9IHtyYW5rfSwgbG9jYWxfcmFuayA9IHtsb2NhbF9yYW5rfSkgdHJhaW5pbmcuLi4iKQogICAgICAgICAgICBwcmludCgiR3B1IERldmljZS'
        'BDb3VudCA6ICIsIG5ncHVzX3Blcl9ub2RlKQogICAgZWxzZToKICAgICAgICBkZXZpY2UgICAgICAgICAgPSB0b3JjaC5kZXZpY2UoJ2N1ZGEn'
        'IGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAnY3B1JykKICAgICAgICBsb2NhbF9yYW5rICAgICAgPSAwCiAgICAgICAgcmFuay'
        'AgICAgICAgICAgID0gMAogICAgY2xhc3NfbmFtZXMsIG51bV9jbGFzc2VzID0gZ2V0X2NsYXNzZXMoY2xhc3Nlc19wYXRoKQogICAgYW5jaG9y'
        'cywgbnVtX2FuY2hvcnMgICAgID0gZ2V0X2FuY2hvcnMoYW5jaG9yc19wYXRoKQogICAgaWYgcHJldHJhaW5lZDoKICAgICAgICBpZiBkaXN0cm'
        'lidXRlZDoKICAgICAgICAgICAgaWYgbG9jYWxfcmFuayA9PSAwOgogICAgICAgICAgICAgICAgZG93bmxvYWRfd2VpZ2h0cygpCiAgICAgICAg'
        'ICAgIGRpc3QuYmFycmllcigpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZG93bmxvYWRfd2VpZ2h0cygpCiAgICBtb2RlbCA9IFlvbG9Cb2'
        'R5KGFuY2hvcnNfbWFzaywgbnVtX2NsYXNzZXMpCiAgICBpZiBub3QgcHJldHJhaW5lZDoKICAgICAgICB3ZWlnaHRzX2luaXQobW9kZWwpCiAg'
        'ICBpZiBtb2RlbF9wYXRoICE9ICcnOgogICAgICAgIGlmIGxvY2FsX3JhbmsgPT0gMDoKICAgICAgICAgICAgcHJpbnQoJ0xvYWQgd2VpZ2h0cy'
        'B7fS4nLmZvcm1hdChtb2RlbF9wYXRoKSkKICAgICAgICBtb2RlbF9kaWN0ICAgICAgPSBtb2RlbC5zdGF0ZV9kaWN0KCkKICAgICAgICBwcmV0'
        'cmFpbmVkX2RpY3QgPSB0b3JjaC5sb2FkKG1vZGVsX3BhdGgsIG1hcF9sb2NhdGlvbiA9IGRldmljZSkKICAgICAgICBsb2FkX2tleSwgbm9fbG'
        '9hZF9rZXksIHRlbXBfZGljdCA9IFtdLCBbXSwge30KICAgICAgICBmb3IgaywgdiBpbiBwcmV0cmFpbmVkX2RpY3QuaXRlbXMoKToKICAgICAg'
        'ICAgICAgaWYgayBpbiBtb2RlbF9kaWN0LmtleXMoKSBhbmQgbnAuc2hhcGUobW9kZWxfZGljdFtrXSkgPT0gbnAuc2hhcGUodik6CiAgICAgIC'
        'AgICAgICAgICB0ZW1wX2RpY3Rba10gPSB2CiAgICAgICAgICAgICAgICBsb2FkX2tleS5hcHBlbmQoaykKICAgICAgICAgICAgZWxzZToKICAg'
        'ICAgICAgICAgICAgIG5vX2xvYWRfa2V5LmFwcGVuZChrKQogICAgICAgIG1vZGVsX2RpY3QudXBkYXRlKHRlbXBfZGljdCkKICAgICAgICBtb2'
        'RlbC5sb2FkX3N0YXRlX2RpY3QobW9kZWxfZGljdCkKICAgICAgICBpZiBsb2NhbF9yYW5rID09IDA6CiAgICAgICAgICAgIHByaW50KCJcblN1'
        'Y2Nlc3NmdWwgTG9hZCBLZXk6Iiwgc3RyKGxvYWRfa2V5KVs6NTAwXSwgIuKApuKAplxuU3VjY2Vzc2Z1bCBMb2FkIEtleSBOdW06IiwgbGVuKG'
        'xvYWRfa2V5KSkKICAgICAgICAgICAgcHJpbnQoIlxuRmFpbCBUbyBMb2FkIEtleToiLCBzdHIobm9fbG9hZF9rZXkpWzo1MDBdLCAi4oCm4oCm'
        'XG5GYWlsIFRvIExvYWQgS2V5IG51bToiLCBsZW4obm9fbG9hZF9rZXkpKQoKICAgICMgRFJNJ3MgYHByb2plY3RgIGNvbnYgaXMgbWVhbnQgdG'
        '8gc3RhcnQgYXMgYW4gZXhhY3QgaWRlbnRpdHkgKHplcm8KICAgICMgd2VpZ2h0ICsgemVybyBiaWFzLCBwZXIgaXRzIG93biBfX2luaXRfXykg'
        'c28gdGhlIHJlc2lkdWFsIGJyYW5jaCBjYW4ndAogICAgIyBwZXJ0dXJiIHRoZSBjb252ZXJnZWQgY2hlY2twb2ludCBvbiBkYXkgb25lLiBCdX'
        'Qgd2VpZ2h0c19pbml0KG1vZGVsKQogICAgIyBhYm92ZSBydW5zIEJFRk9SRSB0aGlzIGNoZWNrcG9pbnQgbG9hZCBhbmQgb3ZlcndyaXRlcyBl'
        'dmVyeSBDb252IGxheWVyCiAgICAjIC0tIGluY2x1ZGluZyBEUk0ncywgd2hpY2ggaGFzIG5vIG1hdGNoaW5nIGtleSBpbiBtb2RlbF9wYXRoIC'
        '0tIHdpdGgKICAgICMgc21hbGwgR2F1c3NpYW4gbm9pc2UgKHN0ZD0wLjAyKSB0aGF0IG5ldmVyIGdldHMgcmVzdG9yZWQuIFJlLXplcm8gaXQK'
        'ICAgICMgZXhwbGljaXRseSBoZXJlLCBhZnRlciB0aGUgbG9hZCwgc28gdGhlIGlkZW50aXR5LWF0LWluaXQgZ3VhcmFudGVlCiAgICAjIGFjdH'
        'VhbGx5IGhvbGRzLgogICAgaWYgJ1VTRV9EUk0nIGluIGRpcigpIGFuZCBVU0VfRFJNIGFuZCBoYXNhdHRyKG1vZGVsLCAnZHJtJykgYW5kIGhh'
        'c2F0dHIobW9kZWwuZHJtLCAncHJvamVjdCcpOgogICAgICAgIG5uLmluaXQuemVyb3NfKG1vZGVsLmRybS5wcm9qZWN0LndlaWdodCkKICAgIC'
        'AgICBubi5pbml0Lnplcm9zXyhtb2RlbC5kcm0ucHJvamVjdC5iaWFzKQogICAgICAgIGlmIGxvY2FsX3JhbmsgPT0gMDoKICAgICAgICAgICAg'
        'cHJpbnQoJ0RSTSBwcm9qZWN0IGxheWVyIHJlLXplcm9lZCBhZnRlciBjaGVja3BvaW50IGxvYWQgKGV4YWN0IGlkZW50aXR5IGF0IGluaXQpLi'
        'cpCgogICAgeW9sb19sb3NzICAgID0gWU9MT0xvc3MoYW5jaG9ycywgbnVtX2NsYXNzZXMsIGlucHV0X3NoYXBlLCBhbmNob3JzX21hc2spCiAg'
        'ICBpZiBsb2NhbF9yYW5rID09IDA6CiAgICAgICAgdGltZV9zdHIgICAgICAgID0gZGF0ZXRpbWUuZGF0ZXRpbWUuc3RyZnRpbWUoZGF0ZXRpbW'
        'UuZGF0ZXRpbWUubm93KCksJyVZXyVtXyVkXyVIXyVNXyVTJykKICAgICAgICBsb2dfZGlyICAgICAgICAgPSBvcy5wYXRoLmpvaW4oc2F2ZV9k'
        'aXIsICJsb3NzXyIgKyBzdHIodGltZV9zdHIpKQogICAgICAgIHByaW50KGYnbG9nX2Rpci17bG9nX2Rpcn0nKQogICAgICAgIGxvc3NfaGlzdG'
        '9yeSAgICA9IExvc3NIaXN0b3J5KGxvZ19kaXIsIG1vZGVsLCBpbnB1dF9zaGFwZT1pbnB1dF9zaGFwZSkKICAgIGVsc2U6CiAgICAgICAgbG9z'
        'c19oaXN0b3J5ICAgID0gTm9uZQogICAgaWYgZnAxNjoKICAgICAgICBmcm9tIHRvcmNoLmN1ZGEuYW1wIGltcG9ydCBHcmFkU2NhbGVyIGFzIE'
        'dyYWRTY2FsZXIKICAgICAgICBzY2FsZXIgPSBHcmFkU2NhbGVyKCkKICAgIGVsc2U6CiAgICAgICAgc2NhbGVyID0gTm9uZQogICAgbW9kZWxf'
        'dHJhaW4gICAgID0gbW9kZWwudHJhaW4oKQogICAgaWYgc3luY19ibiBhbmQgbmdwdXNfcGVyX25vZGUgPiAxIGFuZCBkaXN0cmlidXRlZDoKIC'
        'AgICAgICBtb2RlbF90cmFpbiA9IHRvcmNoLm5uLlN5bmNCYXRjaE5vcm0uY29udmVydF9zeW5jX2JhdGNobm9ybShtb2RlbF90cmFpbikKICAg'
        'IGVsaWYgc3luY19ibjoKICAgICAgICBwcmludCgiU3luY19ibiBpcyBub3Qgc3VwcG9ydCBpbiBvbmUgZ3B1IG9yIG5vdCBkaXN0cmlidXRlZC'
        '4iKQogICAgaWYgQ3VkYToKICAgICAgICBpZiBkaXN0cmlidXRlZDoKICAgICAgICAgICAgbW9kZWxfdHJhaW4gPSBtb2RlbF90cmFpbi5jdWRh'
        'KGxvY2FsX3JhbmspCiAgICAgICAgICAgIG1vZGVsX3RyYWluID0gdG9yY2gubm4ucGFyYWxsZWwuRGlzdHJpYnV0ZWREYXRhUGFyYWxsZWwobW'
        '9kZWxfdHJhaW4sIGRldmljZV9pZHM9W2xvY2FsX3JhbmtdLCBmaW5kX3VudXNlZF9wYXJhbWV0ZXJzPVRydWUpCiAgICAgICAgZWxzZToKICAg'
        'ICAgICAgICAgbW9kZWxfdHJhaW4gPSB0b3JjaC5ubi5EYXRhUGFyYWxsZWwobW9kZWwpCiAgICAgICAgICAgIGN1ZG5uLmJlbmNobWFyayA9IF'
        'RydWUKICAgICAgICAgICAgbW9kZWxfdHJhaW4gPSBtb2RlbF90cmFpbi5jdWRhKCkKICAgIGVtYSA9IE1vZGVsRU1BKG1vZGVsX3RyYWluKQog'
        'ICAgd2l0aCBvcGVuKHRyYWluX2Fubm90YXRpb25fcGF0aCwgZW5jb2Rpbmc9J3V0Zi04JykgYXMgZjoKICAgICAgICB0cmFpbl9saW5lcyA9IG'
        'YucmVhZGxpbmVzKCkKICAgIHdpdGggb3Blbih2YWxfYW5ub3RhdGlvbl9wYXRoLCBlbmNvZGluZz0ndXRmLTgnKSBhcyBmOgogICAgICAgIHZh'
        'bF9saW5lcyAgID0gZi5yZWFkbGluZXMoKQogICAgd2l0aCBvcGVuKGNsZWFyX2Fubm90YXRpb25fcGF0aCwgZW5jb2Rpbmc9J3V0Zi04JykgYX'
        'MgZjoKICAgICAgICBjbGVhcl9saW5lcyA9IGYucmVhZGxpbmVzKCkKICAgIG51bV90cmFpbiAgID0gbGVuKHRyYWluX2xpbmVzKQogICAgbnVt'
        'X3ZhbCAgICAgPSBsZW4odmFsX2xpbmVzKQogICAgaWYgbG9jYWxfcmFuayA9PSAwOgogICAgICAgIHNob3dfY29uZmlnKAogICAgICAgICAgIC'
        'BjbGFzc2VzX3BhdGggPSBjbGFzc2VzX3BhdGgsIGFuY2hvcnNfcGF0aCA9IGFuY2hvcnNfcGF0aCwgYW5jaG9yc19tYXNrID0gYW5jaG9yc19t'
        'YXNrLCBtb2RlbF9wYXRoID0gbW9kZWxfcGF0aCwgaW5wdXRfc2hhcGUgPSBpbnB1dF9zaGFwZSwgXAogICAgICAgICAgICBJbml0X0Vwb2NoID'
        '0gSW5pdF9FcG9jaCwgRnJlZXplX0Vwb2NoID0gRnJlZXplX0Vwb2NoLCBVbkZyZWV6ZV9FcG9jaCA9IFVuRnJlZXplX0Vwb2NoLCBGcmVlemVf'
        'YmF0Y2hfc2l6ZSA9IEZyZWV6ZV9iYXRjaF9zaXplLCBVbmZyZWV6ZV9iYXRjaF9zaXplID0gVW5mcmVlemVfYmF0Y2hfc2l6ZSwgRnJlZXplX1'
        'RyYWluID0gRnJlZXplX1RyYWluLCBcCiAgICAgICAgICAgIEluaXRfbHIgPSBJbml0X2xyLCBNaW5fbHIgPSBNaW5fbHIsIG9wdGltaXplcl90'
        'eXBlID0gb3B0aW1pemVyX3R5cGUsIG1vbWVudHVtID0gbW9tZW50dW0sIGxyX2RlY2F5X3R5cGUgPSBscl9kZWNheV90eXBlLCBcCiAgICAgIC'
        'AgICAgIHNhdmVfcGVyaW9kID0gc2F2ZV9wZXJpb2QsIHNhdmVfZGlyID0gc2F2ZV9kaXIsIG51bV93b3JrZXJzID0gbnVtX3dvcmtlcnMsIG51'
        'bV90cmFpbiA9IG51bV90cmFpbiwgbnVtX3ZhbCA9IG51bV92YWwKICAgICAgICApCiAgICAgICAgd2FudGVkX3N0ZXAgPSA1ZTQgaWYgb3B0aW'
        '1pemVyX3R5cGUgPT0gInNnZCIgZWxzZSAxLjVlNAogICAgICAgIHRvdGFsX3N0ZXAgID0gbnVtX3RyYWluIC8vIFVuZnJlZXplX2JhdGNoX3Np'
        'emUgKiBVbkZyZWV6ZV9FcG9jaAogICAgICAgIGlmIHRvdGFsX3N0ZXAgPD0gd2FudGVkX3N0ZXA6CiAgICAgICAgICAgIGlmIG51bV90cmFpbi'
        'AvLyBVbmZyZWV6ZV9iYXRjaF9zaXplID09IDA6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdUaGUgZGF0YXNldCBpcyB0b28g'
        'c21hbGwgZm9yIHRyYWluaW5nLCBwbGVhc2UgZXhwYW5kIHRoZSBkYXRhc2V0LicpCiAgICBpZiBUcnVlOgogICAgICAgIFVuRnJlZXplX2ZsYW'
        'cgPSBGYWxzZQogICAgICAgIGlmIEZyZWV6ZV9UcmFpbjoKICAgICAgICAgICAgZm9yIHBhcmFtIGluIG1vZGVsLmJhY2tib25lLnBhcmFtZXRl'
        'cnMoKToKICAgICAgICAgICAgICAgIHBhcmFtLnJlcXVpcmVzX2dyYWQgPSBGYWxzZQogICAgICAgIGJhdGNoX3NpemUgPSBGcmVlemVfYmF0Y2'
        'hfc2l6ZSBpZiBGcmVlemVfVHJhaW4gZWxzZSBVbmZyZWV6ZV9iYXRjaF9zaXplCiAgICAgICAgbmJzICAgICAgICAgICAgID0gNjQKICAgICAg'
        'ICBscl9saW1pdF9tYXggICAgPSAxZS0zIGlmIG9wdGltaXplcl90eXBlID09ICdhZGFtJyBlbHNlIDVlLTIKICAgICAgICBscl9saW1pdF9taW'
        '4gICAgPSAzZS00IGlmIG9wdGltaXplcl90eXBlID09ICdhZGFtJyBlbHNlIDFlLTQgICMgbG93ZXJlZCBmcm9tIDVlLTQ6IHRoYXQgZmxvb3Ig'
        'd2FzCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBiaW5kaW'
        '5nIGZvciBvdXIgZmluZS10dW5lIEluaXRfbHIKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAg'
        'ICAgICAgICAgICAgICAgICAjICg8PTJlLTMpLCByZS1oZWF0aW5nIGFuIGFscmVhZHktCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIC'
        'AgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBjb252ZXJnZWQgY2hlY2twb2ludCBtb3JlIHRoYW4KICAgICAgICAg'
        'ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIGludGVuZGVkIC0tIHNlZSB0aG'
        'VzaXMgbm90ZXMuCiAgICAgICAgSW5pdF9scl9maXQgICAgID0gbWluKG1heChiYXRjaF9zaXplIC8gbmJzICogSW5pdF9sciwgbHJfbGltaXRf'
        'bWluKSwgbHJfbGltaXRfbWF4KQogICAgICAgIE1pbl9scl9maXQgICAgICA9IG1pbihtYXgoYmF0Y2hfc2l6ZSAvIG5icyAqIE1pbl9sciwgbH'
        'JfbGltaXRfbWluICogMWUtMiksIGxyX2xpbWl0X21heCAqIDFlLTIpCiAgICAgICAgcGcwLCBwZzEsIHBnMiA9IFtdLCBbXSwgW10KICAgICAg'
        'ICBmb3IgaywgdiBpbiBtb2RlbC5uYW1lZF9tb2R1bGVzKCk6CiAgICAgICAgICAgIGlmIGhhc2F0dHIodiwgImJpYXMiKSBhbmQgaXNpbnN0YW'
        '5jZSh2LmJpYXMsIG5uLlBhcmFtZXRlcik6CiAgICAgICAgICAgICAgICBwZzIuYXBwZW5kKHYuYmlhcykKICAgICAgICAgICAgaWYgaXNpbnN0'
        'YW5jZSh2LCBubi5CYXRjaE5vcm0yZCkgb3IgImJuIiBpbiBrOgogICAgICAgICAgICAgICAgcGcwLmFwcGVuZCh2LndlaWdodCkKICAgICAgIC'
        'AgICAgZWxpZiBoYXNhdHRyKHYsICJ3ZWlnaHQiKSBhbmQgaXNpbnN0YW5jZSh2LndlaWdodCwgbm4uUGFyYW1ldGVyKToKICAgICAgICAgICAg'
        'ICAgIHBnMS5hcHBlbmQodi53ZWlnaHQpCiAgICAgICAgb3B0aW1pemVyID0gewogICAgICAgICAgICAnYWRhbScgIDogb3B0aW0uQWRhbShwZz'
        'AsIEluaXRfbHJfZml0LCBiZXRhcyA9IChtb21lbnR1bSwgMC45OTkpKSwKICAgICAgICAgICAgJ3NnZCcgICA6IG9wdGltLlNHRChwZzAsIElu'
        'aXRfbHJfZml0LCBtb21lbnR1bSA9IG1vbWVudHVtLCBuZXN0ZXJvdj1UcnVlKQogICAgICAgIH1bb3B0aW1pemVyX3R5cGVdCiAgICAgICAgb3'
        'B0aW1pemVyLmFkZF9wYXJhbV9ncm91cCh7InBhcmFtcyI6IHBnMSwgIndlaWdodF9kZWNheSI6IHdlaWdodF9kZWNheX0pCiAgICAgICAgb3B0'
        'aW1pemVyLmFkZF9wYXJhbV9ncm91cCh7InBhcmFtcyI6IHBnMn0pCiAgICAgICAgbHJfc2NoZWR1bGVyX2Z1bmMgPSBnZXRfbHJfc2NoZWR1bG'
        'VyKGxyX2RlY2F5X3R5cGUsIEluaXRfbHJfZml0LCBNaW5fbHJfZml0LCBVbkZyZWV6ZV9FcG9jaCkKICAgICAgICBlcG9jaF9zdGVwICAgICAg'
        'PSBudW1fdHJhaW4gLy8gYmF0Y2hfc2l6ZQogICAgICAgIGlmIGVtYToKICAgICAgICAgICAgZW1hLnVwZGF0ZXMgICAgID0gZXBvY2hfc3RlcC'
        'AqIEluaXRfRXBvY2gKICAgICAgICB0cmFpbl9kYXRhc2V0ICAgPSBZb2xvRGF0YXNldCh0cmFpbl9saW5lcywgY2xlYXJfbGluZXMsIGlucHV0'
        'X3NoYXBlLCBudW1fY2xhc3NlcywgYW5jaG9ycywgYW5jaG9yc19tYXNrLCBlcG9jaF9sZW5ndGg9VW5GcmVlemVfRXBvY2gsIHRyYWluPVRydW'
        'UpCiAgICAgICAgaWYgZGlzdHJpYnV0ZWQ6CiAgICAgICAgICAgIHRyYWluX3NhbXBsZXIgICA9IHRvcmNoLnV0aWxzLmRhdGEuZGlzdHJpYnV0'
        'ZWQuRGlzdHJpYnV0ZWRTYW1wbGVyKHRyYWluX2RhdGFzZXQsIHNodWZmbGU9VHJ1ZSwpCiAgICAgICAgICAgIGJhdGNoX3NpemUgICAgICA9IG'
        'JhdGNoX3NpemUgLy8gbmdwdXNfcGVyX25vZGUKICAgICAgICAgICAgc2h1ZmZsZSAgICAgICAgID0gRmFsc2UKICAgICAgICBlbHNlOgogICAg'
        'ICAgICAgICB0cmFpbl9zYW1wbGVyICAgPSBOb25lCiAgICAgICAgICAgIHNodWZmbGUgICAgICAgICA9IFRydWUKCiAgICAgICAgZ2VuICAgIC'
        'AgICAgICAgID0gRGF0YUxvYWRlcih0cmFpbl9kYXRhc2V0LCBzaHVmZmxlID0gc2h1ZmZsZSwgYmF0Y2hfc2l6ZSA9IGJhdGNoX3NpemUsIG51'
        'bV93b3JrZXJzID0gbnVtX3dvcmtlcnMsIHBpbl9tZW1vcnk9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZHJvcF'
        '9sYXN0PVRydWUsIGNvbGxhdGVfZm49eW9sb19kYXRhc2V0X2NvbGxhdGUsIHNhbXBsZXI9dHJhaW5fc2FtcGxlciwKICAgICAgICAgICAgICAg'
        'ICAgICAgICAgICAgICAgICAgICAgd29ya2VyX2luaXRfZm49cGFydGlhbCh3b3JrZXJfaW5pdF9mbiwgcmFuaz1yYW5rLCBzZWVkPXNlZWQpKQ'
        'oKICAgICAgICBpZiBsb2NhbF9yYW5rID09IDA6CiAgICAgICAgICAgIGV2YWxfY2FsbGJhY2sgICA9IEV2YWxDYWxsYmFjayhtb2RlbCwgaW5w'
        'dXRfc2hhcGUsIGFuY2hvcnMsIGFuY2hvcnNfbWFzaywgY2xhc3NfbmFtZXMsIG51bV9jbGFzc2VzLCB2YWxfbGluZXMsIGxvZ19kaXIsIEN1ZG'
        'EsIFwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBldmFsX2ZsYWc9ZXZhbF9mbGFnLCBwZXJpb2Q9ZXZhbF9w'
        'ZXJpb2QpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZXZhbF9jYWxsYmFjayAgID0gTm9uZQogICAgICAgIGZvciBlcG9jaCBpbiByYW5nZS'
        'hJbml0X0Vwb2NoLCBVbkZyZWV6ZV9FcG9jaCk6CiAgICAgICAgICAgIGlmIGVwb2NoID49IEZyZWV6ZV9FcG9jaCBhbmQgbm90IFVuRnJlZXpl'
        'X2ZsYWcgYW5kIEZyZWV6ZV9UcmFpbjoKICAgICAgICAgICAgICAgIGJhdGNoX3NpemUgPSBVbmZyZWV6ZV9iYXRjaF9zaXplCiAgICAgICAgIC'
        'AgICAgICBuYnMgICAgICAgICAgICAgPSA2NAogICAgICAgICAgICAgICAgbHJfbGltaXRfbWF4ICAgID0gMWUtMyBpZiBvcHRpbWl6ZXJfdHlw'
        'ZSA9PSAnYWRhbScgZWxzZSA1ZS0yCiAgICAgICAgICAgICAgICBscl9saW1pdF9taW4gICAgPSAzZS00IGlmIG9wdGltaXplcl90eXBlID09IC'
        'dhZGFtJyBlbHNlIDFlLTQgICMga2VwdCBpbiBzeW5jIHdpdGggdGhlCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAg'
        'ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIGluaXRpYWwgYmxvY2sgYWJvdmUKICAgICAgICAgICAgICAgIEluaXRfbH'
        'JfZml0ICAgICA9IG1pbihtYXgoYmF0Y2hfc2l6ZSAvIG5icyAqIEluaXRfbHIsIGxyX2xpbWl0X21pbiksIGxyX2xpbWl0X21heCkKICAgICAg'
        'ICAgICAgICAgIE1pbl9scl9maXQgICAgICA9IG1pbihtYXgoYmF0Y2hfc2l6ZSAvIG5icyAqIE1pbl9sciwgbHJfbGltaXRfbWluICogMWUtMi'
        'ksIGxyX2xpbWl0X21heCAqIDFlLTIpCiAgICAgICAgICAgICAgICBscl9zY2hlZHVsZXJfZnVuYyA9IGdldF9scl9zY2hlZHVsZXIobHJfZGVj'
        'YXlfdHlwZSwgSW5pdF9scl9maXQsIE1pbl9scl9maXQsIFVuRnJlZXplX0Vwb2NoKQogICAgICAgICAgICAgICAgZm9yIHBhcmFtIGluIG1vZG'
        'VsLmJhY2tib25lLnBhcmFtZXRlcnMoKToKICAgICAgICAgICAgICAgICAgICBwYXJhbS5yZXF1aXJlc19ncmFkID0gVHJ1ZQogICAgICAgICAg'
        'ICAgICAgZXBvY2hfc3RlcCAgICAgID0gbnVtX3RyYWluIC8vIGJhdGNoX3NpemUKICAgICAgICAgICAgICAgIGVwb2NoX3N0ZXBfdmFsICA9IG'
        '51bV92YWwgLy8gYmF0Y2hfc2l6ZQogICAgICAgICAgICAgICAgaWYgZXBvY2hfc3RlcCA9PSAwIG9yIGVwb2NoX3N0ZXBfdmFsID09IDA6CiAg'
        'ICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGhlIGRhdGFzZXQgaXMgdG9vIHNtYWxsIGZvciB0cmFpbmluZywgcGxlYXNlIG'
        'V4cGFuZCB0aGUgZGF0YXNldC4iKQogICAgICAgICAgICAgICAgaWYgZW1hOgogICAgICAgICAgICAgICAgICAgIGVtYS51cGRhdGVzICAgICA9'
        'IGVwb2NoX3N0ZXAgKiBlcG9jaAogICAgICAgICAgICAgICAgaWYgZGlzdHJpYnV0ZWQ6CiAgICAgICAgICAgICAgICAgICAgYmF0Y2hfc2l6ZS'
        'AgPSBiYXRjaF9zaXplIC8vIG5ncHVzX3Blcl9ub2RlCiAgICAgICAgICAgICAgICBnZW4gICAgICAgICAgICAgPSBEYXRhTG9hZGVyKHRyYWlu'
        'X2RhdGFzZXQsIHNodWZmbGUgPSBzaHVmZmxlLCBiYXRjaF9zaXplID0gYmF0Y2hfc2l6ZSwgbnVtX3dvcmtlcnMgPSBudW1fd29ya2VycywgcG'
        'luX21lbW9yeT1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRyb3BfbGFzdD1UcnVlLCBjb2xsYXRl'
        'X2ZuPXlvbG9fZGF0YXNldF9jb2xsYXRlLCBzYW1wbGVyPXRyYWluX3NhbXBsZXIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIC'
        'AgICAgICAgICAgd29ya2VyX2luaXRfZm49cGFydGlhbCh3b3JrZXJfaW5pdF9mbiwgcmFuaz1yYW5rLCBzZWVkPXNlZWQpKQogICAgICAgICAg'
        'ICAgICAgVW5GcmVlemVfZmxhZyAgID0gVHJ1ZQogICAgICAgICAgICBnZW4uZGF0YXNldC5lcG9jaF9ub3cgICAgICAgPSBlcG9jaAogICAgIC'
        'AgICAgICBpZiBkaXN0cmlidXRlZDoKICAgICAgICAgICAgICAgIHRyYWluX3NhbXBsZXIuc2V0X2Vwb2NoKGVwb2NoKQogICAgICAgICAgICBz'
        'ZXRfb3B0aW1pemVyX2xyKG9wdGltaXplciwgbHJfc2NoZWR1bGVyX2Z1bmMsIGVwb2NoKQogICAgICAgICAgICBmaXRfb25lX2Vwb2NoKG1vZG'
        'VsX3RyYWluLCBtb2RlbCwgZW1hLCB5b2xvX2xvc3MsIGxvc3NfaGlzdG9yeSwgZXZhbF9jYWxsYmFjaywgb3B0aW1pemVyLCBlcG9jaCwgZXBv'
        'Y2hfc3RlcCwgZ2VuLCBVbkZyZWV6ZV9FcG9jaCwgQ3VkYSwgZnAxNiwgc2NhbGVyLCBzYXZlX3BlcmlvZCwgc2F2ZV9kaXIsIGxvY2FsX3Jhbm'
        'spCiAgICAgICAgICAgIGlmIGRpc3RyaWJ1dGVkOgogICAgICAgICAgICAgICAgZGlzdC5iYXJyaWVyKCkKICAgICAgICBpZiBsb2NhbF9yYW5r'
        'ID09IDA6CiAgICAgICAgICAgIGxvc3NfaGlzdG9yeS53cml0ZXIuY2xvc2UoKQo='
    ),
}

for path, b64 in _FILES_B64.items():
    content = base64.b64decode(b64).decode('utf-8')
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    with open(path, 'w') as f:
        f.write(content)
    print(f'Wrote {path} ({len(content)} bytes)')

print('\nAll thesis modification files written -- no git push of the code changes was required.')


In [ ]:
!pip install -q opencv-python-headless tqdm thop
import torch, json, os, sys, glob, time, subprocess

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
n_gpus = torch.cuda.device_count()
print(f"GPU count: {n_gpus}")
for i in range(n_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
if n_gpus < 2:
    print("\nWARNING: this notebook launches training with torchrun --nproc_per_node=2 (DDP)\n"
          "and nets/backbone.py's forward hardcodes an 8+8 split that assumes exactly 2 GPUs.\n"
          "Go to Settings -> Accelerator -> GPU T4 x2 and restart the kernel.")

## Resolve dataset paths

Same resolution logic as `rdfnet_train_eval.ipynb` (VOC-FOG) and `rdfnet_baseline_eval.ipynb`
(RTTS, RDFNet.pth) -- searches for the directory that actually contains the expected
subfolders rather than assuming a fixed nesting depth, since these dataset uploads have
shifted layout before.

In [ ]:
def find_data_root(base, required_subdirs):
    for dirpath, dirnames, _ in os.walk(base):
        if all(d in dirnames for d in required_subdirs):
            return dirpath
    raise FileNotFoundError(f"No directory under {base} contains all of {required_subdirs}")

def find_weight_file(base):
    if os.path.isfile(base) and base.endswith('.pth'):
        return base
    matches = glob.glob(os.path.join(base, '**', '*.pth'), recursive=True)
    if not matches:
        raise FileNotFoundError(f"No .pth weight file found under {base}.")
    return matches[0]

def read_test_ids(test_txt_path):
    return open(test_txt_path).read().strip().split()

# --- VOC-FOG: separate train/ and test/ folders ---
VOC_FOG_BASE = "/kaggle/input/datasets/mdhabibourrahman/voc-fog/VOC_FOG"
TRAIN_ROOT = find_data_root(VOC_FOG_BASE, ("Annotations", "JPEGImages", "VOC2007-FOG"))
TRAIN_FOG_DIR   = os.path.join(TRAIN_ROOT, "VOC2007-FOG")
TRAIN_CLEAN_DIR = os.path.join(TRAIN_ROOT, "JPEGImages")
TRAIN_ANN_DIR   = os.path.join(TRAIN_ROOT, "Annotations")
TEST_ROOT = find_data_root(VOC_FOG_BASE, ("Annotations", "VOCtest-FOG"))
TEST_FOG_DIR = os.path.join(TEST_ROOT, "VOCtest-FOG")
TEST_ANN_DIR = os.path.join(TEST_ROOT, "Annotations")
print(f"VOC-FOG train: fog={TRAIN_FOG_DIR}\n               clean={TRAIN_CLEAN_DIR}\n               ann={TRAIN_ANN_DIR}")
print(f"VOC-FOG test : fog={TEST_FOG_DIR}\n               ann={TEST_ANN_DIR}")

# --- RTTS: flat layout, split defined by ImageSets/Main/test.txt, images are .png ---
RTTS_BASE       = "/kaggle/input/datasets/tuncnguyn/rtts-dataset/RTTS"
RTTS_IMAGES_DIR = os.path.join(RTTS_BASE, "JPEGImages")
RTTS_ANN_DIR    = os.path.join(RTTS_BASE, "Annotations")
RTTS_IDS        = read_test_ids(os.path.join(RTTS_BASE, "ImageSets", "Main", "test.txt"))
print(f"RTTS: {len(RTTS_IDS)} images | images_dir={RTTS_IMAGES_DIR}")

# --- Base paper's converged checkpoint (fine-tune starting point) ---
BASELINE_PTH = find_weight_file("/kaggle/input/datasets/mdhabibourrahman/rdfnet-baseline")
print(f"BASELINE_PTH: {BASELINE_PTH}")

## Build VOC-FOG train/val annotation files

`train.py` reads `train_annotation_path` (foggy) and `clear_annotation_path` (clean), which
must be index-aligned pairs of the same underlying photo, plus `val_annotation_path` (foggy
test images, used both by the periodic `EvalCallback` during training and reused below as the
final VOC-FOG-test set). Identical logic to `rdfnet_train_eval.ipynb`.

In [ ]:
import re
import xml.etree.ElementTree as ET
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

CLASSES_PATH = 'model_data/rtts_classes.txt'
with open(CLASSES_PATH) as f:
    CLASSES = [c.strip() for c in f if c.strip()]
print(f"Classes ({len(CLASSES)}): {CLASSES}")

FOG_SUFFIX_RE = re.compile(r'^(?P<base>.+?)_f\d+$')

def base_stem(stem):
    m = FOG_SUFFIX_RE.match(stem)
    return m.group('base') if m else stem

def is_valid_image(path):
    try:
        with Image.open(path) as im:
            im.load()
        return True
    except (UnidentifiedImageError, OSError):
        return False

def parse_boxes(xml_path):
    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError:
        return None
    boxes = []
    for obj in root.iter('object'):
        diff = obj.find('difficult')
        if diff is not None and int(diff.text) == 1:
            continue
        cls = obj.find('name').text.strip()
        if cls not in CLASSES:
            continue
        bb = obj.find('bndbox')
        coords = [int(float(bb.find(t).text)) for t in ('xmin', 'ymin', 'xmax', 'ymax')]
        boxes.append(','.join(map(str, coords)) + ',' + str(CLASSES.index(cls)))
    return boxes

def box_suffix(boxes):
    return ''.join(' ' + b for b in boxes)

def build_train_pairs(fog_dir, clean_dir, ann_dir, out_fog_path, out_clean_path):
    fog_lines, clean_lines = [], []
    skipped = {'no_clean': 0, 'no_ann': 0, 'no_boxes': 0, 'bad_fog_img': 0, 'bad_clean_img': 0}
    fog_files = sorted(f for f in os.listdir(fog_dir) if f.lower().endswith('.jpg'))
    for fname in tqdm(fog_files, desc='pairing train fog/clean'):
        fog_stem = os.path.splitext(fname)[0]
        stem = base_stem(fog_stem)
        clean_path = os.path.join(clean_dir, stem + '.jpg')
        ann_path = os.path.join(ann_dir, stem + '.xml')
        fog_path = os.path.join(fog_dir, fname)
        if not os.path.exists(clean_path):
            skipped['no_clean'] += 1
            continue
        if not os.path.exists(ann_path):
            skipped['no_ann'] += 1
            continue
        boxes = parse_boxes(ann_path)
        if not boxes:
            skipped['no_boxes'] += 1
            continue
        if not is_valid_image(fog_path):
            skipped['bad_fog_img'] += 1
            continue
        if not is_valid_image(clean_path):
            skipped['bad_clean_img'] += 1
            continue
        suffix = box_suffix(boxes)
        fog_lines.append(fog_path + suffix)
        clean_lines.append(clean_path + suffix)
    with open(out_fog_path, 'w') as f:
        f.write('\n'.join(fog_lines) + '\n')
    with open(out_clean_path, 'w') as f:
        f.write('\n'.join(clean_lines) + '\n')
    print(f"train: {len(fog_lines)} pairs written -> {out_fog_path}, {out_clean_path} | skipped: {skipped}")
    return len(fog_lines)

def build_val(fog_dir, ann_dir, out_path):
    lines, ids = [], []
    skipped = {'no_ann': 0, 'no_boxes': 0, 'bad_img': 0}
    fog_files = sorted(f for f in os.listdir(fog_dir) if f.lower().endswith('.jpg'))
    for fname in tqdm(fog_files, desc='building val/test list'):
        fog_stem = os.path.splitext(fname)[0]
        stem = base_stem(fog_stem)
        ann_path = os.path.join(ann_dir, stem + '.xml')
        fog_path = os.path.join(fog_dir, fname)
        if not os.path.exists(ann_path):
            skipped['no_ann'] += 1
            continue
        boxes = parse_boxes(ann_path)
        if not boxes:
            skipped['no_boxes'] += 1
            continue
        if not is_valid_image(fog_path):
            skipped['bad_img'] += 1
            continue
        lines.append(fog_path + box_suffix(boxes))
        ids.append(fog_stem)
    with open(out_path, 'w') as f:
        f.write('\n'.join(lines) + '\n')
    print(f"val/test: {len(lines)} images written -> {out_path} | skipped: {skipped}")
    return ids

NUM_TRAIN_PAIRS = build_train_pairs(TRAIN_FOG_DIR, TRAIN_CLEAN_DIR, TRAIN_ANN_DIR,
                                     '2007_train_fog.txt', '2007_train.txt')
VOCFOG_TEST_IDS = build_val(TEST_FOG_DIR, TEST_ANN_DIR, '2007_val_fog.txt')
print(f"\nTrain pairs: {NUM_TRAIN_PAIRS} | VOC-FOG test images: {len(VOCFOG_TEST_IDS)}")

## Config writer and fine-tune epoch budget

`write_config(...)` writes `config.py` from scratch for a given run -- this is how the notebook
switches between the baseline / control / proposed architectures and losses, since
`nets/model.py`, `nets/yolo_training.py`, and `utils/utils_fit.py` all read their
`USE_DRM` / `USE_WISE_IOU` / `USE_DWA` toggles from `config.py` at import/construction time.

**`FINETUNE_EPOCHS = 40`**, based on the observed ~5.5 min/epoch from the earlier proposed-run
log (40 epochs x ~5.5 min x 2 runs ~= 7.3h total, comfortably inside a Kaggle session). Both
runs must use the *same* `FINETUNE_EPOCHS` for the control comparison to remain valid -- see
the notebook's markdown intro. If your own timing differs meaningfully, adjust this and restart
from this cell.

**New in this version:** `write_config(...)` also writes the real VOC-FOG-test / RTTS-test
paths, an evaluation period (`real_eval_period`, default 20 epochs), and the baseline mAPs into
`config.py`. During training, `utils/utils_fit.py` uses these to run the *same*
subprocess-isolated `eval_one.py` against the real held-out test sets every `real_eval_period`
epochs (and at the final epoch) and prints the delta against the baseline checkpoint directly in
the training log -- so regressions are visible during training, not just at the very end. Also
fixed since the last run: DRM's residual branch is now explicitly re-zeroed in `train.py` right
after the checkpoint load (undoing noise `weights_init()` would otherwise leave there), and the
SGD learning-rate floor in `train.py` was lowered from `5e-4` to `1e-4` so fine-tuning an
already-converged checkpoint doesn't re-heat the LR further than intended.


In [ ]:
FINETUNE_EPOCHS = 40  # <-- adjust after timing the first epoch; keep IDENTICAL for both runs

def write_config(save_dir, use_drm, use_wiou, use_dwa, model_path=None,
                  freeze_train=False, init_lr=1e-3, unfreeze_epochs=FINETUNE_EPOCHS,
                  eval_period=5, num_workers=2, distributed=True,
                  real_eval_period=20, baseline_vocfog_map=None, baseline_rtts_map=None):
    model_path = model_path or BASELINE_PTH
    config_content = f\'\'\'import os

Cuda            = True
seed            = 114514
distributed     = {distributed}
sync_bn         = False
fp16            = False

classes_path    = 'model_data/rtts_classes.txt'
anchors_path    = 'model_data/yolo_anchors.txt'
anchors_mask    = [[6, 7, 8], [3, 4, 5], [0, 1, 2]]
model_path      = "{model_path}"
input_shape     = [640, 640]

pretrained      = False
Init_Epoch          = 0
Freeze_Epoch        = 100
Freeze_batch_size   = 16
UnFreeze_Epoch      = {unfreeze_epochs}
Unfreeze_batch_size = 16
Freeze_Train        = {freeze_train}
Init_lr             = {init_lr}
Min_lr              = Init_lr * 0.01
optimizer_type      = "sgd"
momentum            = 0.937
weight_decay        = 5e-4
lr_decay_type       = "cos"
save_period         = 10
save_dir            = "{save_dir}"
eval_flag           = True
eval_period         = {eval_period}
num_workers         = {num_workers}

train_annotation_path = "2007_train_fog.txt"
val_annotation_path   = "2007_val_fog.txt"
clear_annotation_path = "2007_train.txt"

USE_DRM         = {use_drm}
USE_WISE_IOU    = {use_wiou}
USE_DWA         = {use_dwa}

# --- Periodic real-test-set evaluation (every real_eval_period epochs, read by
# utils/utils_fit.py during training) -- separate from eval_period above, which
# only evaluates VOC-FOG's own validation split via EvalCallback.
REAL_EVAL_PERIOD       = {real_eval_period}
VOCFOG_TEST_IMAGES_DIR = "{TEST_FOG_DIR}"
VOCFOG_TEST_ANN_DIR    = "{TEST_ANN_DIR}"
RTTS_IMAGES_DIR        = "{RTTS_IMAGES_DIR}"
RTTS_ANN_DIR           = "{RTTS_ANN_DIR}"
RTTS_IDS_FILE          = "{RTTS_TEST_TXT}"
BASELINE_VOCFOG_MAP    = {baseline_vocfog_map!r}
BASELINE_RTTS_MAP      = {baseline_rtts_map!r}
\'\'\'
    with open('config.py', 'w') as f:
        f.write(config_content)
    for k in list(sys.modules.keys()):
        if k == 'config':
            del sys.modules[k]
    import config as cfg
    print(f"config.py written | save_dir={cfg.save_dir} model_path={cfg.model_path}")
    print(f"  USE_DRM={cfg.USE_DRM} USE_WISE_IOU={cfg.USE_WISE_IOU} USE_DWA={cfg.USE_DWA}")
    print(f"  Freeze_Train={cfg.Freeze_Train} Init_lr={cfg.Init_lr} UnFreeze_Epoch={cfg.UnFreeze_Epoch}")
    print(f"  eval_period={cfg.eval_period} num_workers={cfg.num_workers}")
    print(f"  real_eval_period={cfg.REAL_EVAL_PERIOD} baseline_vocfog={cfg.BASELINE_VOCFOG_MAP} baseline_rtts={cfg.BASELINE_RTTS_MAP}")
    return cfg

def run_eval(dataname, model_path, images_dir, ann_dir, image_ext, use_drm, ids_file=None):
    """Rewrites config.py's USE_DRM to match the checkpoint being evaluated, then
    runs eval_one.py as a FRESH subprocess (not an in-kernel import) so the
    architecture nets/model.py builds always matches what this checkpoint was
    trained with -- see eval_one.py's docstring."""
    write_config(save_dir='logs_eval_scratch', use_drm=use_drm, use_wiou=False, use_dwa=False,
                 model_path=model_path)
    out_json = f'/kaggle/working/eval_{dataname}_{os.path.basename(model_path)}.json'
    cmd = ['python', 'eval_one.py', '--dataname', dataname, '--model_path', model_path,
           '--images_dir', images_dir, '--ann_dir', ann_dir, '--image_ext', image_ext,
           '--out_json', out_json]
    if ids_file:
        cmd += ['--ids_file', ids_file]
    print('Running:', ' '.join(cmd))
    ret = subprocess.run(cmd)
    if ret.returncode != 0:
        raise RuntimeError(f"eval_one.py failed for {dataname} / {model_path}")
    with open(out_json) as f:
        return json.load(f)


## Baseline reference (no fine-tuning)

Evaluates the base paper's own `RDFNet.pth` checkpoint as-is, using the same
`eval_one.py` path as the control/proposed checkpoints below, so all three numbers in the
final table come from an identical evaluation pipeline.

In [ ]:
RTTS_TEST_TXT = os.path.join(RTTS_BASE, "ImageSets", "Main", "test.txt")

baseline_vocfog = run_eval('VOC-FOG-test', BASELINE_PTH, TEST_FOG_DIR, TEST_ANN_DIR, '.jpg', use_drm=False)
baseline_rtts   = run_eval('RTTS', BASELINE_PTH, RTTS_IMAGES_DIR, RTTS_ANN_DIR, '.png', use_drm=False,
                            ids_file=RTTS_TEST_TXT)
print(f"\nBaseline  | VOC-FOG mAP={baseline_vocfog['mAP']:.2f}%  RTTS mAP={baseline_rtts['mAP']:.2f}%")

## Control run: unchanged architecture and loss, continued training

Same starting checkpoint, same epoch budget as the proposed run below -- this isolates
"does training longer alone help?" from "does the technique help?" (see notebook intro).

In [ ]:
write_config(save_dir='logs_control', use_drm=False, use_wiou=False, use_dwa=False,
             model_path=BASELINE_PTH, freeze_train=False, init_lr=1e-3,
             real_eval_period=20, baseline_vocfog_map=baseline_vocfog['mAP'],
             baseline_rtts_map=baseline_rtts['mAP'])

t_start = time.time()
ret = os.system(f'torchrun --nproc_per_node={max(torch.cuda.device_count(), 1)} --master_port=12355 train.py')
elapsed = (time.time() - t_start) / 3600
print(f"\n[control] training exit code: {ret} | elapsed: {elapsed:.2f}h")
if ret != 0:
    print("Training did not exit cleanly -- check the log above before proceeding.")


In [ ]:
control_weights = sorted(glob.glob('logs_control/best_epoch_weights.pth')) or sorted(glob.glob('logs_control/last_epoch_weights.pth'))
if not control_weights:
    raise FileNotFoundError("No checkpoint found under logs_control/ -- did training complete at least one epoch?")
CONTROL_PTH = control_weights[-1]
print('Using control weights:', CONTROL_PTH)

control_vocfog = run_eval('VOC-FOG-test', CONTROL_PTH, TEST_FOG_DIR, TEST_ANN_DIR, '.jpg', use_drm=False)
control_rtts   = run_eval('RTTS', CONTROL_PTH, RTTS_IMAGES_DIR, RTTS_ANN_DIR, '.png', use_drm=False,
                           ids_file=RTTS_TEST_TXT)
print(f"\nControl   | VOC-FOG mAP={control_vocfog['mAP']:.2f}%  RTTS mAP={control_rtts['mAP']:.2f}%")

## Proposed run: DRM + Wise-IoU + DWA, continued training

Identical starting checkpoint and epoch budget to the control run above -- only the
architecture (`USE_DRM`) and loss formulation (`USE_WISE_IOU`, `USE_DWA`) differ.

In [ ]:
write_config(save_dir='logs_proposed', use_drm=True, use_wiou=True, use_dwa=True,
             model_path=BASELINE_PTH, freeze_train=False, init_lr=1e-3,
             real_eval_period=20, baseline_vocfog_map=baseline_vocfog['mAP'],
             baseline_rtts_map=baseline_rtts['mAP'])

t_start = time.time()
ret = os.system(f'torchrun --nproc_per_node={max(torch.cuda.device_count(), 1)} --master_port=12355 train.py')
elapsed = (time.time() - t_start) / 3600
print(f"\n[proposed] training exit code: {ret} | elapsed: {elapsed:.2f}h")
if ret != 0:
    print("Training did not exit cleanly -- check the log above before proceeding.")


In [ ]:
proposed_weights = sorted(glob.glob('logs_proposed/best_epoch_weights.pth')) or sorted(glob.glob('logs_proposed/last_epoch_weights.pth'))
if not proposed_weights:
    raise FileNotFoundError("No checkpoint found under logs_proposed/ -- did training complete at least one epoch?")
PROPOSED_PTH = proposed_weights[-1]
print('Using proposed weights:', PROPOSED_PTH)

proposed_vocfog = run_eval('VOC-FOG-test', PROPOSED_PTH, TEST_FOG_DIR, TEST_ANN_DIR, '.jpg', use_drm=True)
proposed_rtts   = run_eval('RTTS', PROPOSED_PTH, RTTS_IMAGES_DIR, RTTS_ANN_DIR, '.png', use_drm=True,
                            ids_file=RTTS_TEST_TXT)
print(f"\nProposed  | VOC-FOG mAP={proposed_vocfog['mAP']:.2f}%  RTTS mAP={proposed_rtts['mAP']:.2f}%")

## Final comparison table

In [ ]:
rows = [
    ('Baseline (paper, 0 extra epochs)', baseline_vocfog, baseline_rtts),
    (f'Control (+{FINETUNE_EPOCHS} epochs, unchanged)', control_vocfog, control_rtts),
    (f'Proposed (+{FINETUNE_EPOCHS} epochs, DRM+WIoU+DWA)', proposed_vocfog, proposed_rtts),
]

print(f"{'Method':42s} {'VOC-FOG mAP':>12s} {'RTTS mAP':>10s}")
for name, vf, rt in rows:
    print(f"{name:42s} {vf['mAP']:11.2f}% {rt['mAP']:9.2f}%")

print("\nPer-class AP on RTTS (bicycle/motorbike are DRM's target classes):")
all_classes = sorted(set(baseline_rtts['per_class_AP']) | set(control_rtts['per_class_AP']) | set(proposed_rtts['per_class_AP']))
print(f"{'class':12s} {'baseline':>10s} {'control':>10s} {'proposed':>10s}")
for c in all_classes:
    b = baseline_rtts['per_class_AP'].get(c, float('nan'))
    ct = control_rtts['per_class_AP'].get(c, float('nan'))
    p = proposed_rtts['per_class_AP'].get(c, float('nan'))
    print(f"{c:12s} {b:9.2f}% {ct:9.2f}% {p:9.2f}%")

print("\nPer-class AP on VOC-FOG-test:")
all_classes_vf = sorted(set(baseline_vocfog['per_class_AP']) | set(control_vocfog['per_class_AP']) | set(proposed_vocfog['per_class_AP']))
print(f"{'class':12s} {'baseline':>10s} {'control':>10s} {'proposed':>10s}")
for c in all_classes_vf:
    b = baseline_vocfog['per_class_AP'].get(c, float('nan'))
    ct = control_vocfog['per_class_AP'].get(c, float('nan'))
    p = proposed_vocfog['per_class_AP'].get(c, float('nan'))
    print(f"{c:12s} {b:9.2f}% {ct:9.2f}% {p:9.2f}%")

## Model complexity (Params/FLOPs) for the control vs. proposed architecture

Confirms how small DRM's added footprint actually is -- everything else (Wise-IoU, DWA) adds
zero parameters since they only change the loss computation, not the network.

In [ ]:
from nets.model import YoloBody
from utils.utils import get_anchors, get_classes
from thop import profile

class_names, num_classes = get_classes('model_data/rtts_classes.txt')
anchors, num_anchors = get_anchors('model_data/yolo_anchors.txt')
anchors_mask = [[6, 7, 8], [3, 4, 5], [0, 1, 2]]
dummy = torch.randn(1, 3, 640, 640)

for label, use_drm in [('control (USE_DRM=False)', False), ('proposed (USE_DRM=True)', True)]:
    write_config(save_dir='logs_eval_scratch', use_drm=use_drm, use_wiou=False, use_dwa=False,
                 model_path=BASELINE_PTH)
    for k in list(sys.modules.keys()):
        if k.startswith('nets.model') or k == 'config':
            del sys.modules[k]
    from nets.model import YoloBody as _YoloBody
    m = _YoloBody(anchors_mask, num_classes)
    m.eval()
    flops, params = profile(m, inputs=(dummy,), verbose=False)
    print(f"{label:28s} Params: {params/1e6:.3f}M  FLOPs: {flops/1e9:.3f}G")

## Save outputs

In [ ]:
import shutil

out_dir = '/kaggle/working/rdfnet_finetune_results'
os.makedirs(out_dir, exist_ok=True)

for pattern in ['logs_control/best_epoch_weights.pth', 'logs_control/last_epoch_weights.pth',
                'logs_control/epoch_map.txt', 'logs_control/epoch_map.png', 'logs_control/epoch_loss.png',
                'logs_proposed/best_epoch_weights.pth', 'logs_proposed/last_epoch_weights.pth',
                'logs_proposed/epoch_map.txt', 'logs_proposed/epoch_map.png', 'logs_proposed/epoch_loss.png',
                'eval_*.json']:
    for f in glob.glob(pattern):
        dest = os.path.join(out_dir, f.replace('/', '_'))
        shutil.copy(f, dest)
        print('Saved:', dest)

summary = {
    'finetune_epochs': FINETUNE_EPOCHS,
    'baseline': {'VOC-FOG': baseline_vocfog, 'RTTS': baseline_rtts},
    'control':  {'VOC-FOG': control_vocfog,  'RTTS': control_rtts},
    'proposed': {'VOC-FOG': proposed_vocfog, 'RTTS': proposed_rtts},
}
with open(os.path.join(out_dir, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\nDone. Download from Kaggle Output tab -> {os.path.basename(out_dir)}/")